# Dataset Inspection & Data-Quality Audit

**Purpose:** answer one question about a dataset before any modelling work starts —

> *What exactly is in this dataset, how reliable and usable is it, what problems does it have,
> and is it worth keeping for further analysis?*

**This notebook never modifies the raw data.** Nothing is cleaned, imputed, interpolated,
renamed, deduplicated, converted or aggregated. Where a temporary transformation is needed for an
inspection (e.g. parsing a date column to measure gaps), the result is stored in a clearly named
*derived* object and the original stays untouched.

**How to use it**

1. Set `DATA_PATH` in the configuration cell below (`.csv`, `.tsv`, `.xlsx`/`.xls`, `.parquet`,
   `.feather`, `.json`, `.nc`/NetCDF).
2. Run all cells from top to bottom.
3. Read the **Data Quality Scorecard** and **Final Dataset Audit** at the end; every statement
   there is backed by evidence produced earlier in the notebook.

**Reading the results** — the notebook deliberately separates:

* **Confirmed problems** — facts (100% missing column, duplicate rows, latitude > 90, infinite values).
* **Potential issues** — things that need domain judgement (statistical outliers, negative values,
  strong correlations, uneven entity coverage, sudden jumps).

A flag is *not* automatically an error.

## 1. Imports and Configuration

**What this section does and why it matters:** loads the few libraries the audit needs and sets the
single configuration point of the notebook (the file path plus a handful of thresholds). Keeping all
settings in one cell means the same notebook can be re-run against any new dataset by editing one line.

In [ ]:
import csv
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# xarray is only required for NetCDF files.
try:
    import xarray as xr
    HAS_XARRAY = True
except ImportError:
    HAS_XARRAY = False

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# ----------------------------------------------------------------- configuration
EXCEL_SHEET = 0            # sheet name or index, used for .xlsx / .xls only
CSV_SEPARATOR = None       # None = pandas default (',' for .csv, '\t' for .tsv)

MAX_PLOT_VARS = 10         # max numeric variables shown in histogram / boxplot grids
MAX_CATEGORY_PLOTS = 6     # max categorical bar charts
NEAR_CONSTANT_SHARE = 0.95 # a column is "near constant" if one value covers >= 95% of rows
HIGH_CARDINALITY_SHARE = 0.5   # unique values >= 50% of rows -> high cardinality
ID_LIKE_SHARE = 0.95       # unique values >= 95% of rows -> ID-like
SAMPLE_FOR_STRING_CHECKS = 20000   # rows sampled for slow string/type checks on huge files
STRONG_CORRELATION = 0.7   # |r| threshold reported as a strong relationship

# ----------------------------------------------------------------- what to inspect
# Every folder listed here is one data source on this machine. Folders *inside* a source are treated
# as separate datasets, so ENTSOE/MonthlyDomesticValues gets its own report.
PROJECT_ROOT = "."
DATA_SOURCES = ["CDS", "EIA", "Ember", "ENTSOE", "IEA", "IRENASTAT", "OWID", "WorldBank"]

# Folders that hold code, caches or previous results - never raw data.
SKIP_DIRS = {"dataaudit", "data_audit_output", "reports", "scripts", "tests", "examples",
             "__pycache__", ".git", ".ipynb_checkpoints", ".venv", "venv", ".idea", "figures"}
# Files that are documentation or link lists rather than data.
SKIP_FILE_PATTERNS = ["how to download", "download_links", "readme"]
DATA_EXTENSIONS = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".xlsm", ".parquet", ".feather",
                   ".json", ".jsonl", ".nc", ".nc4", ".cdf", ".netcdf"}

# "auto" audits the first file of the inventory below; or name one file, e.g.
# DATA_PATH = "OWID/owid-energy-data.csv"
DATA_PATH = "auto"

# ----------------------------------------------------------------- reports / batch mode
REPORT_ROOT = "reports"    # where the saved reports are written
RUN_BATCH = True           # run the full batch (every file of every source) at the end of the notebook
SAVE_TEXT_REPORT = True    # save this single-file audit as a .txt next to the other reports
SUMMARY_JSON_PATH = ""     # set automatically by the batch runner; leave empty for manual runs
BATCH_CHILD = False        # True only while the batch runner executes this notebook for one file

# ----------------------------------------------------------------- very large files
LARGE_FILE_MB = 150        # above this size only the first rows are read, and the audit says so
LARGE_FILE_ROWS = 300_000
JSON_LINES_ROWS = 50_000   # EIA-style .txt bulk files hold one JSON object per line

# A NetCDF grid becomes rows x columns when flattened: 1039 time x 721 lat x 1440 lon is
# 1.08 billion rows and will not fit in memory. Above this many cells the flat table is built
# from a strided SAMPLE of the grid (the time axis is kept complete for as long as possible).
NETCDF_MAX_CELLS = 1_000_000

# Findings are collected here as the notebook runs and are printed as a scorecard in section 23.
findings = []

def note(dimension, status, evidence):
    """Record one scorecard line. status is 'Good', 'Warning' or 'Problem'."""
    findings.append({"quality_dimension": dimension, "status": status, "evidence": evidence})

def name_matches(name, words):
    """True if a keyword occurs in `name` as a whole word (so 'city' does not match 'electricity')."""
    text = str(name).lower()
    return any(re.search(rf"(?<![a-z]){re.escape(w)}(?![a-z])", text) for w in words)

def date_like_share(text_series):
    """Share of values that parse as a PLAUSIBLE date (year 1500-2200).

    Plain numbers are excluded first ("1,234.5" would otherwise become the year 234) and the year
    range filter rejects codes such as "ST-001", which the date parser happily reads as year 1.
    """
    numbers = pd.to_numeric(text_series.str.replace(",", "", regex=False), errors="coerce")
    candidates = text_series[numbers.isna()]
    if len(candidates) == 0:
        return 0.0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")        # silence per-value parsing chatter, not errors
        parsed = pd.to_datetime(candidates, errors="coerce")
    plausible = parsed.notna() & parsed.dt.year.between(1500, 2200)
    return float(plausible.mean() * len(candidates) / len(text_series))

def n_unique(s):
    """nunique() that also works on columns holding unhashable values (lists, dicts, arrays)."""
    try:
        return int(s.nunique(dropna=True))
    except TypeError:
        return int(s.astype(str).nunique(dropna=True))

print("Configuration loaded. Sources to inspect:", ", ".join(DATA_SOURCES))

## 2. Dataset Inventory and Load

**What this section does and why it matters:** first it walks every source folder listed in
`DATA_SOURCES` and builds an **inventory** of the files on this machine — what exists, how big it is,
which files will be audited and which are skipped (archives, documentation, code folders, unsupported
formats) — and flags **byte-identical files**, which is how duplicated downloads such as
`... 2023 (1).csv` show up. The inventory is the map of the whole collection; the rest of the notebook
then audits **one** file from it in detail, and section 25 repeats that audit for every file and saves
one report per dataset folder.

Then the selected file is loaded with the loader matching its extension, and the raw shape, head and
tail are shown. Looking at both ends of a file immediately
exposes classic problems — trailing total rows, footer notes, a repeated header, empty padding rows,
or a file that silently loaded into a single column because the separator was wrong.

Loading details that matter for this collection: the CSV delimiter is sniffed (ENTSO-E exports use
`;`), EIA-style `.txt` bulk files are read as line-delimited JSON, Excel workbooks report their sheet
names, and files larger than `LARGE_FILE_MB` are read only partially — clearly flagged, because an
audit of a truncated file is an audit of a truncated file.

NetCDF files are opened with `xarray`. For the tabular checks further down, a **derived** flat table
(`df`) is created with `Dataset.to_dataframe()`; the original `xarray` object stays available as `ds`.

In [ ]:
import hashlib

project_root = Path(PROJECT_ROOT).resolve()
inventory = pd.DataFrame()
duplicate_files = pd.DataFrame()

if BATCH_CHILD:
    print("# Batch run: the inventory is built once by the runner - skipping it here.")
else:
    records = []
    for source in DATA_SOURCES:
        source_dir = project_root / source
        if not source_dir.exists():
            print(f"! source folder not found, skipped: {source_dir}")
            continue
        for dirpath, dirnames, filenames in os.walk(source_dir):
            dirnames[:] = sorted(d for d in dirnames if d not in SKIP_DIRS)
            for filename in sorted(filenames):
                file_path = Path(dirpath) / filename
                extension = file_path.suffix.lower()
                size_mb = file_path.stat().st_size / 1024**2
                dataset = str(Path(dirpath).relative_to(source_dir))
                if any(pattern in filename.lower() for pattern in SKIP_FILE_PATTERNS):
                    status = "skipped: documentation / link list"
                elif extension == ".zip":
                    status = "skipped: archive (extract it - the extracted folder is audited)"
                elif extension not in DATA_EXTENSIONS:
                    status = f"skipped: not a data format ({extension or 'no extension'})"
                elif size_mb == 0:
                    status = "skipped: empty file"
                else:
                    status = "audit"
                records.append({"source": source,
                                "dataset": "(source root)" if dataset == "." else dataset,
                                "file": filename, "ext": extension,
                                "size_mb": round(size_mb, 3), "status": status,
                                "path": str(file_path)})
    inventory = pd.DataFrame(records)

if len(inventory):
    per_source = inventory.assign(auditable=inventory["status"].eq("audit")).groupby("source").agg(
        files=("file", "size"), to_audit=("auditable", "sum"),
        datasets=("dataset", "nunique"), size_mb=("size_mb", "sum")).round(2)
    display(per_source)
    print("Files found      :", len(inventory))
    print("Files to audit   :", int(inventory["status"].eq("audit").sum()))
    print("Files skipped    :", int((~inventory["status"].eq("audit")).sum()))
    for reason, group in inventory[inventory["status"] != "audit"].groupby("status"):
        print(f"  - {reason}: {len(group)} ({', '.join(group['file'].head(3))}"
              + (" ..." if len(group) > 3 else "") + ")")
elif not BATCH_CHILD:
    print("No data files found. Check PROJECT_ROOT and DATA_SOURCES in the configuration cell.")

In [ ]:
# Byte-identical files: the usual sign of a re-downloaded file such as "... 2023 (1).csv".
if len(inventory):
    to_hash = inventory[inventory["status"] == "audit"].copy()
    digests = []
    for file_path in to_hash["path"]:
        hasher = hashlib.md5()
        with open(file_path, "rb") as handle:
            hasher.update(handle.read(4 * 1024**2))     # first 4 MB is enough to spot copies
        digests.append(hasher.hexdigest())
    to_hash["content_hash"] = digests
    groups = to_hash.groupby(["content_hash", "size_mb"])["path"].apply(list)
    duplicate_files = pd.DataFrame(
        [{"files": len(paths), "size_mb": size, "paths": " | ".join(paths)}
         for (digest, size), paths in groups.items() if len(paths) > 1])

    if len(duplicate_files):
        print("Byte-identical files (kept as they are - only reported):")
        for _, row in duplicate_files.iterrows():
            print(f"  {row['files']} copies, {row['size_mb']} MB each:")
            for path_text in row["paths"].split(" | "):
                print("     ", path_text)
    else:
        print("No byte-identical files found.")

    report_root = Path(REPORT_ROOT)
    report_root.mkdir(parents=True, exist_ok=True)
    inventory.to_csv(report_root / "00_file_inventory.csv", index=False)
    if len(duplicate_files):
        duplicate_files.to_csv(report_root / "00_duplicate_files.csv", index=False)
    print("\nInventory saved to", report_root / "00_file_inventory.csv")
    display(inventory.head(25))
    if len(inventory) > 25:
        print(f"... {len(inventory) - 25} more rows in the saved inventory file.")
else:
    print("# No inventory - skipping the duplicate-file check.")

In [ ]:
# Which file does the detailed audit below run on?
if DATA_PATH == "auto":
    auditable = inventory[inventory["status"] == "audit"] if len(inventory) else pd.DataFrame()
    if len(auditable):
        DATA_PATH = auditable.iloc[0]["path"]
        print("DATA_PATH resolved automatically to the first file of the inventory:")
    else:
        DATA_PATH = "examples/sample_energy_panel.csv"
        print("No data files discovered - falling back to the bundled demo file:")
print("  ", DATA_PATH)
print("\nEvery other file is audited the same way by the batch in section 25.")

In [ ]:
path = Path(DATA_PATH)
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}")

ext = path.suffix.lower()
file_size_mb = path.stat().st_size / 1024**2
ds = None               # xarray.Dataset, only for NetCDF
is_netcdf = False
partial_read = False    # True when the audit describes only part of the file
partial_reason = ""     # how it was reduced - repeated in every report
head_text = ""
loader = ""

row_limit = LARGE_FILE_ROWS if file_size_mb > LARGE_FILE_MB else None

if ext in {".csv", ".tsv", ".txt"}:
    with open(path, "r", encoding="utf-8", errors="replace") as handle:
        head_text = handle.read(65536)

    if head_text.lstrip().startswith("{"):
        # EIA-style bulk export: one JSON object per line.
        df = pd.read_json(path, lines=True, nrows=row_limit or JSON_LINES_ROWS)
        loader = f"pd.read_json(lines=True, nrows={row_limit or JSON_LINES_ROWS})"
        partial_read = len(df) >= (row_limit or JSON_LINES_ROWS)
        if partial_read:
            partial_reason = f"only the first {len(df):,} JSON lines were read"
    else:
        if CSV_SEPARATOR is not None:
            sep = CSV_SEPARATOR
        elif ext == ".tsv":
            sep = "\t"
        else:
            try:
                # ENTSO-E exports are ';' separated, others ',' - let the file decide.
                sep = csv.Sniffer().sniff(head_text[:8192], delimiters=",;\t|").delimiter
            except csv.Error:
                sep = ","
        df = pd.read_csv(path, sep=sep, nrows=row_limit, low_memory=False)
        loader = f"pd.read_csv(sep={sep!r}" + (f", nrows={row_limit})" if row_limit else ")")
        partial_read = row_limit is not None and len(df) == row_limit
        if partial_read:
            partial_reason = (f"only the first {len(df):,} rows of a {file_size_mb:.0f} MB "
                              "file were read")

elif ext in {".xlsx", ".xls", ".xlsm"}:
    sheets = pd.ExcelFile(path).sheet_names
    print("Sheets in this workbook:", sheets)
    if len(sheets) > 1:
        print(f"NOTE: only sheet {EXCEL_SHEET!r} is audited. Set EXCEL_SHEET to inspect another one.")
    df = pd.read_excel(path, sheet_name=EXCEL_SHEET, nrows=row_limit)
    loader = f"pd.read_excel(sheet_name={EXCEL_SHEET!r})"
    partial_read = row_limit is not None and len(df) == row_limit
    if partial_read:
        partial_reason = f"only the first {len(df):,} rows of sheet {EXCEL_SHEET!r} were read"

elif ext == ".parquet":
    df = pd.read_parquet(path)
    loader = "pd.read_parquet()"
elif ext == ".feather":
    df = pd.read_feather(path)
    loader = "pd.read_feather()"
elif ext in {".json", ".jsonl"}:
    df = pd.read_json(path, lines=(ext == ".jsonl"))
    loader = "pd.read_json()"
elif ext in {".nc", ".nc4", ".cdf", ".netcdf"}:
    if not HAS_XARRAY:
        raise ImportError("xarray is required to read NetCDF files (pip install xarray netCDF4).")
    is_netcdf = True
    ds = xr.open_dataset(path)          # lazy: the values are not read into memory yet
    loader = "xr.open_dataset() -> to_dataframe()"

    # Flattening a grid multiplies its size (1039 time x 721 lat x 1440 lon = 1.08 billion rows),
    # so the flat table is built from a strided SAMPLE when the grid is too large. Spatial
    # dimensions are thinned first so the time axis stays complete and the temporal checks
    # further down stay meaningful. `ds` always keeps the full, untouched dataset.
    TIME_LIKE_DIMS = {"time", "valid_time", "date", "datetime", "step", "forecast_time",
                      "month", "year", "season"}
    grid_cells = 1
    for size in ds.sizes.values():
        grid_cells *= int(size)
    strides = {dim: 1 for dim in ds.sizes}

    def sampled_cells(current):
        total = 1
        for dim, size in ds.sizes.items():
            total *= int(np.ceil(size / current[dim]))
        return total

    while sampled_cells(strides) > NETCDF_MAX_CELLS:
        thinnable = [d for d in ds.sizes
                     if d not in TIME_LIKE_DIMS and np.ceil(ds.sizes[d] / strides[d]) > 1]
        if not thinnable:      # nothing spatial left to thin - only then touch the time axis
            thinnable = [d for d in ds.sizes if np.ceil(ds.sizes[d] / strides[d]) > 1]
        if not thinnable:
            break
        widest = max(thinnable, key=lambda d: np.ceil(ds.sizes[d] / strides[d]))
        strides[widest] *= 2

    ds_flat = ds
    if any(step > 1 for step in strides.values()):
        ds_flat = ds.isel({dim: slice(None, None, step)
                           for dim, step in strides.items() if step > 1})

    # String coordinates (ERA5 stores "expver" as 4-character text) are expanded to one string
    # per grid point when flattened - that is what exhausts memory. They are reported in the
    # metadata section below instead of being carried into the table.
    string_coords = [name for name, coord in ds_flat.coords.items()
                     if coord.dtype.kind in {"U", "S", "O"} and name not in ds_flat.dims]
    if string_coords:
        ds_flat = ds_flat.drop_vars(string_coords)

    try:
        df = ds_flat.to_dataframe().reset_index()
    except MemoryError:
        # Emergency fallback: thin every dimension hard rather than lose the audit.
        strides = {dim: max(1, step) * 8 for dim, step in strides.items()}
        print("MemoryError while flattening - retrying with a much coarser sample:", strides)
        ds_flat = ds.isel({dim: slice(None, None, step) for dim, step in strides.items()})
        keep_out = [c for c in string_coords if c in ds_flat.coords]
        if keep_out:
            ds_flat = ds_flat.drop_vars(keep_out)
        df = ds_flat.to_dataframe().reset_index()

    thinned = {dim: step for dim, step in strides.items() if step > 1}
    if thinned or string_coords:
        partial_read = bool(thinned)
        pieces = []
        if thinned:
            pieces.append("grid subsampled (" +
                          ", ".join(f"every {step}. {dim}" for dim, step in thinned.items()) +
                          f") - {len(df):,} of {grid_cells:,} grid points "
                          f"({len(df) / grid_cells:.2%})")
        if string_coords:
            pieces.append("text coordinates kept out of the flat table: " + ", ".join(string_coords))
        partial_reason = "; ".join(pieces)
else:
    raise ValueError(f"Unsupported file extension: {ext}")

print("File            :", path.name)
print("Folder          :", path.parent)
print("File type       :", ext, "(NetCDF)" if is_netcdf else "")
print("File size       : %.2f MB" % file_size_mb)
print("Loaded with     :", loader)
print("Shape (rows, columns):", df.shape)

if partial_read:
    print("\n*** PARTIAL READ ***")
    print(" ", partial_reason)
    print("  Every result below describes THAT SAMPLE, not the whole file "
          "(row counts, coverage and duplicates in particular).")
elif partial_reason:
    print("\nNote:", partial_reason)

if df.shape[1] == 1 and any(d in head_text for d in [";", "\t", "|"]):
    print("\nWARNING: the file loaded into a single column although the first bytes contain other "
          "delimiters - the separator is probably wrong. Set CSV_SEPARATOR and re-run.")

if is_netcdf:
    print("\nNote: `df` is a DERIVED flat table built from the NetCDF dataset; `ds` holds the original.")

In [ ]:
display(df.head())

In [ ]:
display(df.tail())

In [ ]:
# Compact raw representation - useful to spot odd column names, padding and stray unnamed columns.
print(repr(df).split("\n\n")[0][:2000])
print("\nColumn names as loaded:")
for i, c in enumerate(df.columns):
    print(f"  [{i}] {c!r}")

In [ ]:
# NetCDF only: original dataset summary, dimensions, coordinates, variables and attributes.
if is_netcdf:
    print(ds)
else:
    print("# Not a NetCDF file - skipping xarray dataset summary.")

## 3. Dataset Structure

**What this section does and why it matters:** establishes the physical shape of the data — how many
rows and columns, what the storage types are, how much memory it needs, and how many columns fall into
each type family. Storage types drive everything that follows: a date stored as text or a number stored
as a string will silently break statistics, sorting and joins.

In [ ]:
df.info(memory_usage="deep")

In [ ]:
numeric_cols  = df.select_dtypes(include="number").columns.tolist()
bool_cols     = df.select_dtypes(include="bool").columns.tolist()
datetime_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
cat_cols      = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()
other_cols    = [c for c in df.columns
                 if c not in numeric_cols + bool_cols + datetime_cols + cat_cols]

print("Rows                     :", f"{len(df):,}")
print("Columns                  :", df.shape[1])
print("Memory usage             : %.2f MB" % (df.memory_usage(deep=True).sum() / 1024**2))
print("Numeric columns          :", len(numeric_cols), numeric_cols[:15])
print("Categorical/object columns:", len(cat_cols), cat_cols[:15])
print("Datetime columns         :", len(datetime_cols), datetime_cols)
print("Boolean columns          :", len(bool_cols), bool_cols)
print("Other dtypes             :", len(other_cols), other_cols)

if partial_read:
    note("Dataset size", "Warning",
         f"{len(df):,} rows x {df.shape[1]} columns AUDITED FROM A SAMPLE - {partial_reason}.")
elif len(df) == 0:
    note("Dataset size", "Problem", "The file loaded with 0 rows.")
elif len(df) < 100:
    note("Dataset size", "Warning", f"Only {len(df):,} rows x {df.shape[1]} columns "
         f"({file_size_mb:.2f} MB) - very small dataset.")
else:
    note("Dataset size", "Good", f"{len(df):,} rows x {df.shape[1]} columns ({file_size_mb:.2f} MB).")

In [ ]:
# Compact per-column overview: dtype, completeness and distinct values.
structure = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "non_null_count": df.notna().sum().to_numpy(),
    "missing_%": (df.isna().mean().to_numpy() * 100).round(2),
    "unique_values": [n_unique(df[c]) for c in df.columns],
})
display(structure)

In [ ]:
# NetCDF only: dimensions, coordinates, variable metadata, time and spatial coverage.
netcdf_notes = []      # findings that only exist in NetCDF metadata, reused in section 20

if is_netcdf:
    print("DIMENSIONS")
    for name, size in ds.sizes.items():
        print(f"  {name:20s} size = {size}")

    print("\nCOORDINATES")
    for name, coord in ds.coords.items():
        values = np.asarray(coord.values).ravel()
        if values.size == 0:
            rng = "empty"
        elif values.dtype.kind in {"U", "S", "O"}:
            # Text coordinates (ERA5 "expver") have no range - list what they contain.
            distinct = pd.unique(values)
            rng = f"{len(distinct)} distinct text value(s): {list(distinct[:5])}"
        else:
            rng = f"{values.min()} -> {values.max()}"
        print(f"  {name:20s} shape={coord.shape} dtype={coord.dtype} range: {rng}")
        for key in ("units", "long_name", "standard_name", "calendar", "axis"):
            if key in coord.attrs:
                print(f"      {key}: {coord.attrs[key]}")

    print("\nDATA VARIABLES")
    for name, var in ds.data_vars.items():
        print(f"  {name}")
        print(f"      dims={var.dims} shape={var.shape} dtype={var.dtype}")
        for key in ("units", "long_name", "standard_name", "description", "cell_methods",
                    "_FillValue", "missing_value", "scale_factor", "add_offset"):
            if key in var.attrs:
                print(f"      {key}: {var.attrs[key]}")

    # A text coordinate with more than one value means the file mixes data streams or versions
    # (ERA5: expver 0001 = final reanalysis, 0005 = preliminary ERA5T). Worth knowing before use.
    for name, coord in ds.coords.items():
        if coord.dtype.kind in {"U", "S", "O"}:
            distinct = pd.unique(np.asarray(coord.values).ravel())
            if len(distinct) > 1:
                message = (f"text coordinate '{name}' holds {len(distinct)} different values "
                           f"{[str(v) for v in distinct[:5]]} - the file mixes several data "
                           "streams/versions; check which records belong to which before using them")
                netcdf_notes.append(message)
                print(f"\n  ATTENTION: {message}")

    print("\nGLOBAL ATTRIBUTES")
    for key, value in ds.attrs.items():
        print(f"  {key}: {str(value)[:200]}")

    # Time and spatial coverage straight from the coordinates, when present.
    for tname in [c for c in ds.coords if str(c).lower() in {"time", "date", "valid_time"}]:
        tvals = pd.to_datetime(np.asarray(ds[tname].values).ravel(), errors="coerce")
        print(f"\nTime coverage ({tname}): {tvals.min()} -> {tvals.max()}  "
              f"({len(tvals)} steps, calendar={ds[tname].attrs.get('calendar', 'not documented')})")
    for cname in [c for c in ds.coords if str(c).lower() in
                  {"lat", "latitude", "lon", "longitude", "y", "x"}]:
        vals = np.asarray(ds[cname].values, dtype="float64").ravel()
        print(f"Spatial coverage ({cname}): {np.nanmin(vals)} -> {np.nanmax(vals)} ({vals.size} points)")
else:
    print("# Not a NetCDF file - skipping dimension/coordinate/attribute report.")

## 4. Important Variables

**What this section does and why it matters:** scans column names (and NetCDF metadata) for terms that
often mark the substantive content of a dataset — demand, generation, population, GDP, temperature,
country, date, and so on. It also collects the *entity-like* and *time-like* column names that later
sections need.

> This is a **name-based inspection aid only**. A column called `demand` is not proof that it contains
> demand; every hit below is reported as *"potentially relevant based on column/variable name"* and
> must be confirmed against documentation.

In [ ]:
KEYWORDS = {
    "demand / load":        ["demand", "load", "consumption", "consumed", "usage"],
    "generation / supply":  ["generation", "generated", "supply", "output", "production", "capacity"],
    "energy carriers":      ["electricity", "power", "energy", "solar", "wind", "hydro", "nuclear",
                             "coal", "gas", "oil", "renewable", "fossil", "biomass"],
    "socio-economic":       ["population", "gdp", "income", "price", "cost", "employment", "urban"],
    "weather / climate":    ["temperature", "temp", "precipitation", "rain", "snow", "humidity",
                             "radiation", "irradiance", "pressure", "cloud", "wind_speed"],
    "emissions":            ["co2", "emission", "ghg", "carbon", "intensity"],
    "geography / entity":   ["country", "iso", "region", "state", "province", "city", "station",
                             "zone", "area", "site", "company", "entity", "lat", "lon",
                             "latitude", "longitude"],
    "time":                 ["date", "time", "year", "month", "day", "hour", "week", "quarter",
                             "period", "timestamp"],
}

flagged = {}
for topic, words in KEYWORDS.items():
    hits = [c for c in df.columns if name_matches(c, words)]
    if hits:
        flagged[topic] = hits

print("Potentially relevant columns based on column names only "
      "(NOT semantic validation - confirm against documentation):\n")
for topic, hits in flagged.items():
    print(f"{topic}:")
    for c in hits:
        print(f"  - {c}")
    print()

important_cols = sorted({c for hits in flagged.values() for c in hits}, key=list(df.columns).index)
if not flagged:
    print("No column names matched the keyword list - the dataset may use codes or "
          "non-English names, so variable meaning has to be taken from documentation.")

In [ ]:
# Columns whose NAMES suggest an entity identifier or a time reference.
ENTITY_HINTS = ["country", "iso", "region", "state", "province", "city", "station", "site",
                "company", "entity", "zone", "sector", "plant", "node", "name", "code", "id"]
TIME_HINTS   = ["date", "time", "year", "month", "day", "hour", "week", "quarter", "period", "stamp"]

entity_name_cols = [c for c in df.columns
                    if name_matches(c, ENTITY_HINTS) and not name_matches(c, TIME_HINTS)]
time_name_cols   = [c for c in df.columns if name_matches(c, TIME_HINTS)]

print("Entity-like column names :", entity_name_cols or "none")
print("Time-like column names   :", time_name_cols or "none")

In [ ]:
# NetCDF only: the same keyword scan against variable names, long_name and standard_name.
if is_netcdf:
    print("Potentially relevant NetCDF variables (name/metadata match only):\n")
    any_hit = False
    for name, var in ds.variables.items():
        text = " ".join([str(name),
                         str(var.attrs.get("long_name", "")),
                         str(var.attrs.get("standard_name", ""))]).lower()
        topics = [t for t, words in KEYWORDS.items() if name_matches(text, words)]
        if topics:
            any_hit = True
            print(f"  {name}  ->  {', '.join(topics)}")
            print(f"      long_name: {var.attrs.get('long_name', 'not documented')}")
            print(f"      units    : {var.attrs.get('units', 'not documented')}")
    if not any_hit:
        print("  No NetCDF variable name or metadata matched the keyword list.")
else:
    print("# Not a NetCDF file - skipping metadata keyword scan.")

## 5. Missing Data Analysis

**What this section does and why it matters:** quantifies exactly where the holes are — per column,
overall, and per row. Missingness decides whether a variable is usable at all, and a headline row count
can hide the fact that the one column you need is 60% empty.

> Missing data is **not automatically "bad"**. A sparse column may be irrelevant to your question, and
> a small gap in a long series may not matter. The impact depends entirely on the intended use.

In [ ]:
missing_count = df.isna().sum()
missing_table = pd.DataFrame({
    "column": missing_count.index,
    "missing_count": missing_count.to_numpy(),
    "non_missing_count": df.notna().sum().to_numpy(),
    "missing_%": (df.isna().mean().to_numpy() * 100).round(2),
}).sort_values("missing_%", ascending=False).reset_index(drop=True)

total_cells = int(df.size)
overall_missing_pct = (df.isna().sum().sum() / total_cells * 100) if total_cells else 0.0
cols_with_missing = int((missing_count > 0).sum())
rows_with_missing = int(df.isna().any(axis=1).sum())

display(missing_table)
print("Overall missing cells      : %.2f%% (%s of %s cells)"
      % (overall_missing_pct, f"{int(df.isna().sum().sum()):,}", f"{total_cells:,}"))
print("Columns containing missing :", cols_with_missing, "of", df.shape[1])
print("Rows with >=1 missing value: %s (%.2f%% of rows)"
      % (f"{rows_with_missing:,}", rows_with_missing / len(df) * 100 if len(df) else 0))

# A fully empty column makes every row "incomplete", so the same count is shown without such columns.
non_empty_cols = [c for c in df.columns if df[c].notna().any()]
rows_missing_excl_empty = int(df[non_empty_cols].isna().any(axis=1).sum()) if non_empty_cols else 0
print("  ... ignoring fully empty columns: %s (%.2f%% of rows)"
      % (f"{rows_missing_excl_empty:,}",
         rows_missing_excl_empty / len(df) * 100 if len(df) else 0))

In [ ]:
# Flag the severity bands. Interpretation still depends on how the column will be used.
bands = [(100, "100% missing (column is empty - confirmed problem)"),
         (75,  "> 75% missing (barely usable)"),
         (50,  "> 50% missing"),
         (25,  "> 25% missing"),
         (10,  "> 10% missing")]

reported = set()
for threshold, label in bands:
    if threshold == 100:
        hits = missing_table[missing_table["missing_%"] >= 100]
    else:
        hits = missing_table[(missing_table["missing_%"] > threshold)
                             & (~missing_table["column"].isin(reported))]
    if not hits.empty:
        print(f"\n{label}:")
        for _, r in hits.iterrows():
            print(f"  - {r['column']}: {r['missing_%']:.2f}% ({int(r['missing_count']):,} values)")
        reported.update(hits["column"])

if missing_count.sum() == 0:
    print("No missing values anywhere in the dataset.")
elif not reported:
    print("\nNo column exceeds 10% missing.")

max_missing = float(missing_table["missing_%"].max()) if len(missing_table) else 0.0
empty_cols = missing_table.loc[missing_table["missing_%"] >= 100, "column"].tolist()
if empty_cols:
    note("Missing data", "Problem",
         f"{len(empty_cols)} fully empty column(s): {', '.join(map(str, empty_cols[:5]))}; "
         f"overall {overall_missing_pct:.2f}% of cells missing.")
elif max_missing > 25:
    note("Missing data", "Warning",
         f"Overall {overall_missing_pct:.2f}% of cells missing; worst column "
         f"{missing_table.iloc[0]['column']} at {max_missing:.2f}%.")
else:
    note("Missing data", "Good",
         f"Overall {overall_missing_pct:.2f}% of cells missing; worst column at {max_missing:.2f}%.")

In [ ]:
# Bar chart only if there is something to show.
plot_missing = missing_table[missing_table["missing_%"] > 0]
if plot_missing.empty:
    print("# No missing data - skipping missing-data plot.")
else:
    top = plot_missing.head(30).iloc[::-1]
    plt.figure(figsize=(8, max(3, 0.32 * len(top))))
    plt.barh(top["column"].astype(str), top["missing_%"], color="#c0504d")
    plt.xlabel("Missing (%)")
    plt.title(f"Missing values by column (top {len(top)})")
    plt.tight_layout()
    plt.show()

## 6. Duplicate Records

**What this section does and why it matters:** duplicated rows inflate counts, bias averages and break
joins. Two things are checked: fully identical rows, and repeated combinations of identifier-like
columns (e.g. the same `country` + `date` appearing twice), which is usually the more damaging kind
because it is invisible in a row count.

> Duplicates are reported, **never removed**. Whether a repeated key is an error depends on the grain
> the dataset is supposed to have.

In [ ]:
try:
    dup_mask = df.duplicated(keep=False)          # all members of each duplicate group
    dup_first = df.duplicated()                   # extra copies only
except TypeError:
    # Fallback when cells contain unhashable values (lists/arrays); a string view is comparable.
    dup_mask = df.astype(str).duplicated(keep=False)
    dup_first = df.astype(str).duplicated()

n_dup_rows = int(dup_first.sum())
dup_pct = n_dup_rows / len(df) * 100 if len(df) else 0.0

print("Fully duplicated rows (extra copies):", f"{n_dup_rows:,}", "(%.2f%% of rows)" % dup_pct)
print("Rows involved in a duplicate group  :", f"{int(dup_mask.sum()):,}")

if n_dup_rows:
    print("\nExample duplicated rows (first 10, not removed):")
    display(df[dup_mask].sort_values(list(df.columns)[:3]).head(10))

In [ ]:
# Repeated identifier combinations. Only combinations that plausibly define the grain are tested.
key_candidates = [c for c in (entity_name_cols + time_name_cols) if c in df.columns]
key_candidates = [c for c in key_candidates if 1 < n_unique(df[c]) <= len(df)]

duplicate_key_report = []

for c in key_candidates:                          # single identifier columns
    d = int(df.duplicated(subset=[c]).sum())
    duplicate_key_report.append({"key": c, "duplicate_rows": d,
                                 "duplicate_%": round(d / len(df) * 100, 2) if len(df) else 0.0})

for e in entity_name_cols:                        # entity + time combinations
    for t in time_name_cols:
        if e in df.columns and t in df.columns:
            d = int(df.duplicated(subset=[e, t]).sum())
            duplicate_key_report.append({"key": f"{e} + {t}", "duplicate_rows": d,
                                         "duplicate_%": round(d / len(df) * 100, 2) if len(df) else 0.0})

if duplicate_key_report:
    key_table = (pd.DataFrame(duplicate_key_report)
                   .sort_values("duplicate_rows")
                   .reset_index(drop=True))
    display(key_table)
    print("A duplicate count of 0 means the combination is unique in this file; it does NOT prove")
    print("the combination is the intended key. Non-zero counts on entity+time combinations are the")
    print("most likely candidates for a real grain violation.")
else:
    key_table = pd.DataFrame(columns=["key", "duplicate_rows", "duplicate_%"])
    print("# No identifier-like columns detected - skipping key-combination duplicate check.")

if n_dup_rows:
    note("Duplicates", "Problem", f"{n_dup_rows:,} fully duplicated rows ({dup_pct:.2f}%).")
else:
    note("Duplicates", "Good", "No fully duplicated rows.")

## 7. Constant / Low-Variation Columns

**What this section does and why it matters:** a column with a single value carries no information, and
a near-constant column carries almost none. Beyond being useless as a feature, these columns often
reveal a *loading* problem: a filter applied upstream, metadata pasted into a data column, a failed
export, or a sensor stuck on one reading.

Nothing is deleted — the columns are only listed with the evidence.

In [ ]:
variation_rows = []
for c in df.columns:
    s = df[c]
    non_null = s.dropna()
    if non_null.empty:
        variation_rows.append({"column": c, "unique_values": 0, "most_common_value": None,
                               "most_common_%": np.nan, "flag": "empty (100% missing)"})
        continue
    try:
        counts = non_null.value_counts()
    except TypeError:
        counts = non_null.astype(str).value_counts()
    top_value = counts.index[0]
    top_share = counts.iloc[0] / len(df) * 100
    uniq = n_unique(s)
    if uniq <= 1:
        flag = "constant"
    elif top_share >= NEAR_CONSTANT_SHARE * 100:
        flag = "near-constant"
    else:
        flag = ""
    variation_rows.append({"column": c, "unique_values": uniq,
                           "most_common_value": str(top_value)[:40],
                           "most_common_%": round(top_share, 2), "flag": flag})

variation = pd.DataFrame(variation_rows)
constant_cols = variation.loc[variation["flag"] == "constant", "column"].tolist()
near_constant_cols = variation.loc[variation["flag"] == "near-constant", "column"].tolist()

display(variation.sort_values("most_common_%", ascending=False).reset_index(drop=True))

print("Constant columns      :", constant_cols or "none")
print("Near-constant columns (>= %.0f%% one value): %s"
      % (NEAR_CONSTANT_SHARE * 100, near_constant_cols or "none"))
print("\nPossible explanations: unusable feature, upstream filter, metadata stored as data,")
print("failed export, or genuinely no variation in this sample. Columns are kept as-is.")

if constant_cols:
    note("Constant columns", "Warning",
         f"{len(constant_cols)} constant column(s): {', '.join(map(str, constant_cols[:6]))}"
         + (f"; {len(near_constant_cols)} near-constant." if near_constant_cols else "."))
elif near_constant_cols:
    note("Constant columns", "Warning",
         f"{len(near_constant_cols)} near-constant column(s): {', '.join(map(str, near_constant_cols[:6]))}.")
else:
    note("Constant columns", "Good", "No constant or near-constant columns.")

## 8. Cardinality

**What this section does and why it matters:** the number of distinct values in a non-numeric column
tells you what it *is*: a small set of categories, a code list, a free-text field, or a row identifier.
This drives how the column can be used (grouping key, category, join key) and highlights columns that
should probably not be treated as features at all.

High cardinality is **not** an error — timestamps, UUIDs and transaction numbers are legitimately unique.

In [ ]:
non_numeric = [c for c in df.columns if c not in numeric_cols]
if not non_numeric:
    cardinality = pd.DataFrame(columns=["column", "dtype", "unique_values", "unique_%", "classification"])
    print("# No non-numeric columns - skipping cardinality analysis.")
else:
    rows = []
    for c in non_numeric:
        uniq = n_unique(df[c])
        share = uniq / len(df) if len(df) else 0
        if uniq <= 1:
            kind = "constant"
        elif share >= ID_LIKE_SHARE and uniq > 20:
            kind = "ID-like (nearly unique per row)"
        elif share >= HIGH_CARDINALITY_SHARE and uniq > 20:
            kind = "high cardinality"
        elif uniq <= 20:
            kind = "low-cardinality categorical"
        else:
            kind = "medium cardinality"
        rows.append({"column": c, "dtype": str(df[c].dtype), "unique_values": uniq,
                     "unique_%": round(share * 100, 2), "classification": kind})
    cardinality = pd.DataFrame(rows).sort_values("unique_values", ascending=False).reset_index(drop=True)
    display(cardinality)

    id_like_cols = cardinality.loc[
        cardinality["classification"].str.startswith("ID-like"), "column"].tolist()
    low_card_cols = cardinality.loc[
        cardinality["classification"] == "low-cardinality categorical", "column"].tolist()
    print("Low-cardinality categoricals (usable as groups):", low_card_cols or "none")
    print("Potential identifier columns                  :", id_like_cols or "none")
    print("\nID-like and high-cardinality columns are flagged for awareness, not as errors")
    print("(IDs, timestamps, UUIDs and URLs are unique by design).")

## 9. Data Type Validation

**What this section does and why it matters:** the most common silent data-quality failure is a value
stored in the wrong type — numbers held as text because of thousands separators or an embedded unit,
dates held as strings, booleans written as `"yes"`/`"no"`, and placeholder tokens (`"N/A"`, `"-"`,
`"null"`) that pandas did not recognise as missing. Any of these turns a numeric column into an object
column and quietly disables every statistic computed on it.

Nothing is converted here — the findings are reported so you can decide the correct fix later.

In [ ]:
PLACEHOLDERS = {"", "na", "n/a", "n.a.", "nan", "null", "none", "nil", "-", "--", "---",
                "?", ".", "..", "missing", "unknown", "#n/a", "#na", "n/d", "no data"}
BOOL_TOKENS = {"true", "false", "yes", "no", "y", "n", "t", "f", "0", "1"}
NUMBER_WITH_UNIT = re.compile(r"^[-+]?[\d.,\s]+\s*[a-zA-Z%°/€$£]+\.?$")

type_issues = []

for c in cat_cols:
    s = df[c].dropna()
    if s.empty:
        continue
    sample = s.sample(SAMPLE_FOR_STRING_CHECKS, random_state=0) if len(s) > SAMPLE_FOR_STRING_CHECKS else s
    text = sample.astype(str).str.strip()          # temporary copy for inspection only
    lower = text.str.lower()

    numeric_like = pd.to_numeric(text.str.replace(",", "", regex=False),
                                 errors="coerce").notna().mean()
    date_like = date_like_share(text)

    py_types = sorted({type(v).__name__ for v in sample.head(5000)})
    placeholder_hits = int(lower.isin(PLACEHOLDERS).sum())
    empty_hits = int((text == "").sum())
    unit_hits = int(text.str.match(NUMBER_WITH_UNIT).sum())
    whitespace_hits = int((sample.astype(str) != text).sum())

    if numeric_like >= 0.9 and not set(lower.unique()) <= BOOL_TOKENS:
        type_issues.append({"column": c, "issue": "numeric values stored as text",
                            "evidence": f"{numeric_like:.0%} of non-null values parse as numbers"})
    if date_like >= 0.9:
        type_issues.append({"column": c, "issue": "dates stored as text",
                            "evidence": f"{date_like:.0%} of non-null values parse as dates"})
    if set(lower.unique()) <= BOOL_TOKENS and n_unique(s) <= 3:
        type_issues.append({"column": c, "issue": "boolean values stored as text",
                            "evidence": f"values: {sorted(lower.unique())}"})
    if len(py_types) > 1:
        type_issues.append({"column": c, "issue": "mixed Python types in one column",
                            "evidence": f"types found: {py_types}"})
    if placeholder_hits:
        type_issues.append({"column": c, "issue": "missing-value placeholders stored as text",
                            "evidence": f"{placeholder_hits:,} cells such as "
                                        f"{sorted(set(lower[lower.isin(PLACEHOLDERS)]))[:5]}"})
    if empty_hits:
        type_issues.append({"column": c, "issue": "empty strings (not counted as NaN)",
                            "evidence": f"{empty_hits:,} cells"})
    if unit_hits and unit_hits / len(text) > 0.5:
        type_issues.append({"column": c, "issue": "numbers carrying units or separators",
                            "evidence": f"{unit_hits:,} cells like {text[text.str.match(NUMBER_WITH_UNIT)].head(3).tolist()}"})
    if whitespace_hits:
        type_issues.append({"column": c, "issue": "leading/trailing whitespace",
                            "evidence": f"{whitespace_hits:,} cells"})

# Numeric columns holding only whole numbers, and year-like integers: informational, not errors.
for c in numeric_cols:
    s = df[c].dropna()
    if s.empty:
        continue
    if "year" in str(c).lower() and pd.api.types.is_integer_dtype(df[c]):
        type_issues.append({"column": c, "issue": "year stored as integer (informational)",
                            "evidence": f"range {int(s.min())}-{int(s.max())}"})

type_issue_table = pd.DataFrame(type_issues)
if type_issue_table.empty:
    print("No data-type problems detected in object/text columns.")
    note("Data types", "Good", "No numeric-as-text, date-as-text, mixed-type or placeholder issues found.")
else:
    display(type_issue_table)
    real = type_issue_table[~type_issue_table["issue"].str.contains("informational")]
    for _, r in real.iterrows():
        print(f'Potential issue: column "{r["column"]}" - {r["issue"]} ({r["evidence"]}).')
    if real.empty:
        note("Data types", "Good", "Only informational type remarks.")
    else:
        note("Data types", "Warning",
             f"{len(real)} type issue(s), e.g. " +
             "; ".join(f'{r["column"]}: {r["issue"]}' for _, r in real.head(3).iterrows()))

## 10. Numeric Variable Statistics

**What this section does and why it matters:** the standard five-number summary plus a few additions
that matter for auditing — the range, the coefficient of variation, and the count of zeros and negative
values. Zeros and negatives are highlighted because for some variables they are impossible
(population, installed capacity), for others perfectly normal (net load, temperature in °C, price).

> No value is called an error here. Validity depends on what the variable actually measures.

In [ ]:
if not numeric_cols:
    numeric_stats = pd.DataFrame()
    print("# No numeric columns - skipping numeric statistics.")
else:
    describe = df[numeric_cols].describe().T
    extra = pd.DataFrame({
        "range": df[numeric_cols].max(numeric_only=True) - df[numeric_cols].min(numeric_only=True),
        "coef_of_variation": (df[numeric_cols].std(numeric_only=True)
                              / df[numeric_cols].mean(numeric_only=True).replace(0, np.nan)),
        "zeros": (df[numeric_cols] == 0).sum(),
        "negatives": (df[numeric_cols] < 0).sum(),
        "missing": df[numeric_cols].isna().sum(),
        "infinite": np.isinf(df[numeric_cols].to_numpy(dtype="float64", na_value=np.nan)).sum(axis=0),
    })
    numeric_stats = describe.join(extra).round(4)
    display(numeric_stats)

    zero_heavy = numeric_stats[numeric_stats["zeros"] > 0.5 * len(df)].index.tolist()
    with_negatives = numeric_stats[numeric_stats["negatives"] > 0].index.tolist()
    print("Columns where zeros cover >50% of rows (may be real, may be encoded missing):",
          zero_heavy or "none")
    print("Columns containing negative values (check whether negative is meaningful here):",
          with_negatives or "none")

## 11. Categorical Variables

**What this section does and why it matters:** shows what the text/categorical columns actually contain
— the distinct value count and the most frequent values with their share. This is where you notice
inconsistent spellings, unexpected sentinel categories, and a dominant category that swamps everything
else. Bar charts are drawn only for columns with at most 20 distinct values; plotting a free-text or
ID column produces nothing readable.

In [ ]:
plot_ready = []
if not cat_cols:
    print("# No categorical/object columns - skipping categorical analysis.")
else:
    for c in cat_cols[:30]:
        s = df[c].dropna()
        if s.empty:
            print(f"--- {c}: 100% missing, no values to summarise\n")
            continue
        try:
            counts = s.value_counts()
        except TypeError:
            counts = s.astype(str).value_counts()
        top = counts.head(5)
        summary = pd.DataFrame({"value": top.index.astype(str),
                                "frequency": top.to_numpy(),
                                "%_of_rows": (top.to_numpy() / len(df) * 100).round(2)})
        print(f"--- {c}  |  unique values: {n_unique(s):,}  |  missing: {df[c].isna().sum():,}")
        display(summary)
        if 1 < len(counts) <= 20:
            plot_ready.append(c)
    if len(cat_cols) > 30:
        print(f"({len(cat_cols) - 30} further categorical columns not printed to keep the output readable.)")

In [ ]:
selected = plot_ready[:MAX_CATEGORY_PLOTS]
if not selected:
    print("# No categorical column with <=20 distinct values - skipping bar charts.")
else:
    ncols = min(2, len(selected))
    nrows = int(np.ceil(len(selected) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, selected):
        counts = df[c].astype(str).value_counts().head(20)
        ax.barh(counts.index[::-1], counts.to_numpy()[::-1], color="#4f81bd")
        ax.set_title(c)
        ax.set_xlabel("count")
    for ax in axes[len(selected):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    if len(plot_ready) > MAX_CATEGORY_PLOTS:
        print(f"({len(plot_ready) - MAX_CATEGORY_PLOTS} further low-cardinality columns not plotted.)")

## 12. Temporal Coverage

**What this section does and why it matters:** establishes *when* the data applies, at what frequency,
and whether the calendar is complete. Period coverage is usually more decisive than row count: a file
with a million rows is still unusable for a monthly study if half the months are absent or the sampling
frequency changes half way through.

Date columns stored as text are parsed into a **derived** series (`time_parsed`) purely for this
analysis. Missing periods are reported, **never interpolated**.

In [ ]:
# Detect columns that hold a usable time reference.
time_candidates = []

for c in datetime_cols:
    time_candidates.append((c, df[c]))

for c in df.columns:
    if c in datetime_cols or c in [name for name, _ in time_candidates]:
        continue
    if c in cat_cols:
        s = df[c].dropna().astype(str)
        if s.empty:
            continue
        sample = s.sample(min(len(s), 5000), random_state=0)
        # Codes such as "ST-001" and plain numbers must not be mistaken for dates.
        if date_like_share(sample) >= 0.9:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                time_candidates.append((c, pd.to_datetime(df[c], errors="coerce")))

# year (+ month/day) stored as numbers -> build a derived timestamp
lower_names = {str(c).lower(): c for c in df.columns}
year_col = lower_names.get("year")
month_col = lower_names.get("month")
day_col = lower_names.get("day")
if year_col is not None and year_col in numeric_cols:
    parts = pd.DataFrame({"year": pd.to_numeric(df[year_col], errors="coerce")})
    label = str(year_col)
    parts["month"] = pd.to_numeric(df[month_col], errors="coerce") if month_col in (numeric_cols or []) else 1
    if month_col is not None and month_col in numeric_cols:
        label += f" + {month_col}"
    parts["day"] = pd.to_numeric(df[day_col], errors="coerce") if (day_col is not None and day_col in numeric_cols) else 1
    if day_col is not None and day_col in numeric_cols:
        label += f" + {day_col}"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        derived_time = pd.to_datetime(parts, errors="coerce")
    if derived_time.notna().mean() >= 0.9:
        time_candidates.append((f"{label} (derived timestamp)", derived_time))

print("Time-like columns detected:", [c for c, _ in time_candidates] or "none")

In [ ]:
def describe_frequency(values):
    """Return (label, median_gap) for a sorted DatetimeIndex of unique timestamps."""
    if len(values) < 3:
        return "unknown (fewer than 3 timestamps)", None
    diffs = pd.Series(values).diff().dropna()
    med = diffs.median()
    days = med / pd.Timedelta(days=1)
    if days <= 0:
        label = "unknown"
    elif abs(days - 1 / 24) < 0.005:
        label = "hourly"
    elif days < 0.04:
        label = "sub-hourly"
    elif days < 0.9:
        label = "intra-daily"
    elif days < 1.5:
        label = "daily"
    elif days < 9:
        label = "weekly" if abs(days - 7) < 1.5 else "irregular (multi-day)"
    elif days < 45:
        label = "monthly" if 27 <= days <= 32 else "irregular (multi-week)"
    elif days < 150:
        label = "quarterly" if 85 <= days <= 95 else "irregular (multi-month)"
    elif days < 400:
        label = "yearly" if 360 <= days <= 370 else "irregular (multi-month)"
    else:
        label = "multi-year steps"
    # Calendar months and years have unequal lengths, so compare with a 20% tolerance.
    tolerance = 0.2 * med
    regular_share = ((diffs - med).abs() <= tolerance).mean()
    if regular_share < 0.9:
        label += f" but irregular ({regular_share:.0%} of steps within 20% of the median gap)"
    return label, med

FREQ_ALIAS = {"hourly": "h", "daily": "D", "weekly": "W", "monthly": "MS",
              "quarterly": "QS", "yearly": "YS"}

time_report = []
for name, series in time_candidates:
    values = pd.DatetimeIndex(series.dropna().unique()).sort_values()
    if len(values) == 0:
        continue
    label, med = describe_frequency(values)
    duplicated_stamps = int(series.dropna().duplicated().sum())
    time_report.append({
        "time_column": name,
        "min": values.min(),
        "max": values.max(),
        "unique_timestamps": len(values),
        "records": int(series.notna().sum()),
        "unparsed/missing": int(series.isna().sum()),
        "inferred_frequency": label,
        "duplicate_timestamps": duplicated_stamps,
    })

if not time_report:
    time_table = pd.DataFrame()
    main_time_col, time_parsed = None, None
    print("# No datetime column detected - skipping temporal analysis.")
    note("Temporal coverage", "Not applicable", "No time column detected.")
else:
    time_table = pd.DataFrame(time_report)
    display(time_table)
    # The column with the most distinct timestamps is used as the main time axis below.
    best = time_table.sort_values("unique_timestamps", ascending=False).iloc[0]["time_column"]
    main_time_col = best
    time_parsed = dict((n, s) for n, s in time_candidates)[best]   # DERIVED, original untouched
    print("Main time axis used for the remaining checks:", main_time_col)

In [ ]:
missing_periods = []
expected_periods = observed_periods = None
freq_label = "unknown"

if main_time_col is None:
    print("# No time column - skipping calendar completeness check.")
else:
    values = pd.DatetimeIndex(time_parsed.dropna().unique()).sort_values()
    freq_label, med_gap = describe_frequency(values)
    base = freq_label.split(" but ")[0]
    alias = FREQ_ALIAS.get(base)

    print(f"Time range        : {values.min()}  ->  {values.max()}")
    print(f"Inferred frequency: {freq_label}")
    print(f"Unique timestamps : {len(values):,}")

    if alias is None:
        print("Frequency is irregular or unknown - a complete-calendar check would be misleading,")
        print("so only the observed gaps are reported below.")
    else:
        full = pd.date_range(values.min(), values.max(), freq=alias)
        if base in {"monthly", "quarterly", "yearly"}:
            period_freq = {"monthly": "M", "quarterly": "Q", "yearly": "Y"}[base]
            expected = pd.PeriodIndex(full, freq=period_freq)
            observed = pd.PeriodIndex(values, freq=period_freq)
        else:
            expected, observed = full, values
        expected_periods = len(expected)
        observed_periods = len(pd.Index(observed).unique())
        missing_periods = [str(p) for p in pd.Index(expected).difference(pd.Index(observed))]
        print(f"Expected periods  : {expected_periods:,}")
        print(f"Observed periods  : {observed_periods:,}")
        print(f"Missing periods   : {len(missing_periods):,}")
        if missing_periods:
            print("  first 20 missing:", missing_periods[:20])
            print("  (reported only - no interpolation is performed)")

    # Observed gaps larger than 1.5x the typical step.
    diffs = pd.Series(values).diff().dropna()
    if med_gap is not None and len(diffs):
        gaps = diffs[diffs > 1.5 * med_gap]
        print(f"\nGaps larger than 1.5x the median step: {len(gaps)}")
        for idx in gaps.sort_values(ascending=False).head(10).index:
            print(f"  {values[idx-1]} -> {values[idx]}  ({diffs[idx]})")

    dup_stamps = int(time_parsed.dropna().duplicated().sum())
    print(f"\nRows sharing a timestamp with another row: {dup_stamps:,}")
    print("(expected in panel data with several entities per period; a problem in a single series)")

    if missing_periods:
        note("Temporal coverage", "Warning",
             f"{freq_label} data {values.min().date()} -> {values.max().date()}; "
             f"{len(missing_periods)} of {expected_periods} expected periods missing.")
    else:
        note("Temporal coverage", "Good",
             f"{freq_label} data {values.min().date()} -> {values.max().date()}; "
             f"{observed_periods if observed_periods else len(values)} periods, no calendar gaps.")

## 13. Entity Coverage

**What this section does and why it matters:** for panel data (countries, regions, stations, companies)
the row count says almost nothing. What matters is how many entities exist, how many observations each
one has, and whether they share the same time span. Unbalanced panels are common and legitimate, but
they change what analysis is possible — and an average computed over an unbalanced panel is dominated
by the best-covered entities.

Coverage differences are reported per entity; nothing is dropped or filled.

In [ ]:
# Pick the most plausible entity column: identifier-like name, repeated values, not unique per row.
entity_candidates = []
for c in entity_name_cols:
    uniq = n_unique(df[c])
    if len(df) and 1 < uniq < max(2, 0.9 * len(df)):
        entity_candidates.append((c, uniq))

PREFERRED = ("country", "iso", "entity", "region", "station", "site", "city", "company")
entity_candidates.sort(key=lambda x: (not str(x[0]).lower().startswith(PREFERRED), x[1]))

main_entity_col = entity_candidates[0][0] if entity_candidates else None
print("Entity column candidates:", entity_candidates or "none")
print("Entity column used      :", main_entity_col or "none")

In [ ]:
entity_time_table = pd.DataFrame()
n_entities = None
balance = "not applicable"

if main_entity_col is None:
    print("# No entity/identifier column detected - skipping entity coverage analysis.")
    note("Entity coverage", "Not applicable", "No entity identifier column detected.")
else:
    per_entity = df[main_entity_col].value_counts(dropna=False)
    n_entities = int(df[main_entity_col].nunique(dropna=True))
    print(f"Unique entities in '{main_entity_col}': {n_entities:,}")
    print(f"Missing entity identifiers          : {int(df[main_entity_col].isna().sum()):,}")
    print(f"Records per entity - min {per_entity.min():,} | median {per_entity.median():,.0f} | "
          f"mean {per_entity.mean():,.1f} | max {per_entity.max():,}")

    spread = per_entity.max() - per_entity.min()
    cv = per_entity.std() / per_entity.mean() if per_entity.mean() else 0
    if spread == 0:
        balance = "balanced (identical record count for every entity)"
    elif cv < 0.25:
        balance = f"moderately unbalanced (CV of records per entity = {cv:.2f})"
    else:
        balance = f"highly unbalanced (CV of records per entity = {cv:.2f})"
    print("Panel balance:", balance)

    print("\nEntities with the fewest records:")
    display(per_entity.tail(10).to_frame("records"))

    plt.figure(figsize=(7, 4))
    plt.hist(per_entity.to_numpy(), bins=min(30, max(5, n_entities)), color="#4f81bd", edgecolor="white")
    plt.xlabel(f"records per {main_entity_col}")
    plt.ylabel("number of entities")
    plt.title(f"Distribution of observations per {main_entity_col}")
    plt.tight_layout()
    plt.show()

    if balance.startswith("balanced"):
        note("Entity coverage", "Good", f"{n_entities} entities in '{main_entity_col}', {balance}.")
    elif balance.startswith("moderately"):
        note("Entity coverage", "Warning", f"{n_entities} entities in '{main_entity_col}', {balance}.")
    else:
        note("Entity coverage", "Warning",
             f"{n_entities} entities in '{main_entity_col}', {balance}; "
             f"records per entity range {per_entity.min():,}-{per_entity.max():,}.")

In [ ]:
# Per-entity temporal coverage (only when both an entity and a time axis exist).
if main_entity_col is None or main_time_col is None:
    print("# Entity and/or time column missing - skipping per-entity temporal coverage.")
else:
    # DERIVED helper frame; the original df is not modified.
    coverage_input = pd.DataFrame({"entity": df[main_entity_col].astype(str),
                                   "time": time_parsed}).dropna(subset=["time"])
    grouped = coverage_input.groupby("entity")["time"]
    entity_time_table = pd.DataFrame({
        "first_date": grouped.min(),
        "last_date": grouped.max(),
        "records": grouped.size(),
        "unique_periods": grouped.nunique(),
    })

    base = freq_label.split(" but ")[0]
    alias = FREQ_ALIAS.get(base)
    if alias is not None:
        period_freq = {"hourly": "h", "daily": "D", "weekly": "W",
                       "monthly": "M", "quarterly": "Q", "yearly": "Y"}[base]
        global_expected = pd.PeriodIndex(
            pd.date_range(coverage_input["time"].min(), coverage_input["time"].max(), freq=alias),
            freq=period_freq)
        missing_counts, own_gap_counts = [], []
        for entity, sub in coverage_input.groupby("entity")["time"]:
            observed = pd.PeriodIndex(pd.DatetimeIndex(sub.unique()), freq=period_freq).unique()
            missing_counts.append(len(global_expected.difference(observed)))
            own_span = pd.period_range(observed.min(), observed.max(), freq=period_freq)
            own_gap_counts.append(len(own_span.difference(observed)))
        entity_time_table["missing_periods_vs_full_range"] = missing_counts
        entity_time_table["gaps_inside_own_range"] = own_gap_counts

    entity_time_table = entity_time_table.sort_values("records")
    display(entity_time_table.head(15))
    if len(entity_time_table) > 15:
        print(f"... {len(entity_time_table) - 15} more entities (sorted ascending by record count).")

    if "missing_periods_vs_full_range" in entity_time_table:
        short = entity_time_table[entity_time_table["missing_periods_vs_full_range"] > 0]
        gappy = entity_time_table[entity_time_table["gaps_inside_own_range"] > 0]
        print(f"\nEntities not covering the full time range: {len(short)} of {len(entity_time_table)}")
        print(f"Entities with gaps inside their own range: {len(gappy)} of {len(entity_time_table)}")
        if len(gappy):
            print("  e.g.", ", ".join(gappy.index[:5]))

## 14. Geographic Coverage

**What this section does and why it matters:** geographic keys are the usual join surface between
datasets, and they are the usual place where joins fail. Country names come in several spellings
(`US`, `USA`, `United States`), ISO codes come with wrong lengths, and coordinates come out of range.
Out-of-range coordinates are *confirmed* errors; naming variants are *potential* harmonisation issues
and are flagged, never merged.

In [ ]:
GEO_HINTS = ["country", "iso", "region", "state", "province", "city", "station", "nation", "area", "zone"]
geo_cols = [c for c in df.columns if name_matches(c, GEO_HINTS)]
lat_cols = [c for c in df.columns if str(c).lower() in
            {"lat", "latitude", "y_lat", "lat_deg"} or "latitude" in str(c).lower()]
lon_cols = [c for c in df.columns if str(c).lower() in
            {"lon", "long", "longitude", "lng", "x_lon"} or "longitude" in str(c).lower()]

geo_issues = []

if not geo_cols and not lat_cols and not lon_cols:
    print("# No geographic columns detected - skipping geographic coverage analysis.")
else:
    print("Geographic columns:", geo_cols or "none")
    print("Latitude columns  :", lat_cols or "none")
    print("Longitude columns :", lon_cols or "none")

    for c in geo_cols:
        values = df[c].dropna().astype(str)
        uniq = sorted(values.unique())
        print(f"\n--- {c}: {len(uniq):,} distinct values, {int(df[c].isna().sum()):,} missing")
        print("   ", uniq[:25], "..." if len(uniq) > 25 else "")
        if df[c].isna().any():
            geo_issues.append(f"{c}: {int(df[c].isna().sum()):,} missing identifiers")

        # Same label written differently (case, spacing, punctuation) - possible harmonisation issue.
        normalised = values.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
        variants = pd.DataFrame({"raw": values, "norm": normalised}).drop_duplicates()
        clash = variants.groupby("norm")["raw"].nunique()
        clash = clash[clash > 1]
        for norm in clash.index[:10]:
            forms = variants.loc[variants["norm"] == norm, "raw"].unique().tolist()
            print(f"    possible harmonisation issue: {forms}")
            geo_issues.append(f"{c}: inconsistent spellings {forms}")

        # ISO-code shaped columns: check the expected code length.
        if "iso" in str(c).lower() or str(c).lower() in {"code", "country_code"}:
            lengths = values.str.len().value_counts()
            if len(lengths) > 1:
                print(f"    mixed code lengths: {lengths.to_dict()} (ISO2 and ISO3 mixed?)")
                geo_issues.append(f"{c}: mixed code lengths {lengths.to_dict()}")
            bad = values[~values.str.fullmatch(r"[A-Za-z]{2,3}")]
            if len(bad):
                print(f"    {len(bad):,} values are not 2-3 letter codes, e.g. {bad.unique()[:5].tolist()}")
                geo_issues.append(f"{c}: {len(bad):,} non-alphabetic codes")

In [ ]:
# Known-alias check across name columns (flag only - nothing is merged).
ALIAS_GROUPS = [
    {"united states", "united states of america", "usa", "us", "u.s.", "u.s.a."},
    {"united kingdom", "uk", "great britain", "britain", "gb"},
    {"russia", "russian federation"},
    {"south korea", "korea, rep.", "republic of korea", "korea rep"},
    {"north korea", "korea, dem. people's rep.", "dpr korea"},
    {"czechia", "czech republic"},
    {"turkey", "türkiye", "turkiye"},
    {"ivory coast", "cote d'ivoire", "côte d'ivoire"},
    {"netherlands", "the netherlands", "holland"},
    {"china", "people's republic of china", "prc", "mainland china"},
]

name_like_geo = [c for c in geo_cols if c in cat_cols]
if not name_like_geo:
    print("# No text geographic columns - skipping alias check.")
else:
    for c in name_like_geo:
        present = set(df[c].dropna().astype(str).str.strip().str.lower().unique())
        for group in ALIAS_GROUPS:
            overlap = sorted(present & group)
            if len(overlap) > 1:
                print(f"{c}: possible duplicate identifiers for one country -> {overlap}")
                geo_issues.append(f"{c}: alias variants {overlap}")
    print("Alias check finished (variants are flagged as possible harmonisation issues, not merged).")

In [ ]:
# Coordinate bounds: these ARE mathematically invalid when out of range.
invalid_coords = 0
for c in lat_cols:
    s = pd.to_numeric(df[c], errors="coerce")
    bad = int(((s < -90) | (s > 90)).sum())
    invalid_coords += bad
    print(f"{c}: range {s.min()} -> {s.max()} | values outside [-90, 90]: {bad:,}"
          + ("  <-- CONFIRMED INVALID" if bad else ""))
for c in lon_cols:
    s = pd.to_numeric(df[c], errors="coerce")
    bad = int(((s < -180) | (s > 180)).sum())
    invalid_coords += bad
    print(f"{c}: range {s.min()} -> {s.max()} | values outside [-180, 180]: {bad:,}"
          + ("  <-- CONFIRMED INVALID" if bad else ""))
if not lat_cols and not lon_cols:
    print("# No latitude/longitude columns - skipping coordinate range check.")

## 15. Variable Distributions

**What this section does and why it matters:** a histogram shows in one glance what summary statistics
hide — bimodality, heavy skew, a spike at zero, a sentinel value such as `-9999`, or a variable that is
really a category coded as a number. At most ten variables are plotted, chosen from the non-constant
numeric columns and preferring the ones flagged as potentially relevant in section 4.

In [ ]:
plot_numeric = [c for c in numeric_cols if c not in constant_cols and df[c].notna().any()]

# Time components and coordinates are structural, not measurements: keep them for last.
structural_cols = [c for c in df.columns
                   if str(c).lower() in {"year", "month", "day", "hour", "minute", "week", "quarter"}
                   or c in lat_cols or c in lon_cols]

# Preference order: structural columns last, then flagged as relevant, then relative variability.
ranked = sorted(
    plot_numeric,
    key=lambda c: (c in structural_cols,
                   c not in important_cols,
                   -abs(float(numeric_stats.loc[c, "coef_of_variation"]))
                   if (len(numeric_stats) and c in numeric_stats.index
                       and pd.notna(numeric_stats.loc[c, "coef_of_variation"])) else 0.0)
)
selected_numeric = ranked[:MAX_PLOT_VARS]

if not selected_numeric:
    print("# No non-constant numeric columns - skipping distribution plots.")
else:
    print("Plotted variables:", selected_numeric)
    if len(plot_numeric) > MAX_PLOT_VARS:
        print(f"({len(plot_numeric) - MAX_PLOT_VARS} further numeric columns not plotted; "
              "selection prefers flagged and high-variability variables.)")
    ncols = min(3, len(selected_numeric))
    nrows = int(np.ceil(len(selected_numeric) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.1 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, selected_numeric):
        values = df[c].replace([np.inf, -np.inf], np.nan).dropna()
        ax.hist(values, bins=min(50, max(10, n_unique(df[c]))), color="#4f81bd", edgecolor="white")
        ax.set_title(c, fontsize=10)
        ax.set_ylabel("count")
    for ax in axes[len(selected_numeric):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 16. Outlier Detection

**What this section does and why it matters:** applies the standard IQR rule
(`Q1 - 1.5*IQR`, `Q3 + 1.5*IQR`) to every numeric variable and counts how many observations fall
outside it. This is a cheap way to find sentinel codes, unit mix-ups and data-entry slips.

> **Potential statistical outliers detected. Requires domain validation.** The IQR rule flags points
> that are far from the middle of the distribution; on skewed data (energy demand, GDP, precipitation)
> a large flagged share is normal and does not mean the values are wrong.

In [ ]:
if not plot_numeric:
    outlier_table = pd.DataFrame()
    print("# No non-constant numeric columns - skipping outlier analysis.")
else:
    rows = []
    for c in plot_numeric:
        s = df[c].replace([np.inf, -np.inf], np.nan).dropna()
        if s.empty:
            continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = (s < low) | (s > high)
        rows.append({"variable": c, "lower_bound": round(float(low), 4),
                     "upper_bound": round(float(high), 4),
                     "outlier_count": int(mask.sum()),
                     "outlier_%": round(mask.mean() * 100, 2),
                     "min": float(s.min()), "max": float(s.max())})
    outlier_table = pd.DataFrame(rows).sort_values("outlier_%", ascending=False).reset_index(drop=True)
    display(outlier_table)
    heavy = outlier_table[outlier_table["outlier_%"] > 5]
    print("Potential statistical outliers detected. Requires domain validation.")
    if len(heavy):
        print("Variables with >5% of values outside the IQR fence "
              "(often simply a skewed distribution):", heavy["variable"].tolist())
        note("Outliers", "Warning",
             f"{len(heavy)} variable(s) with >5% IQR-outliers, e.g. "
             f"{heavy.iloc[0]['variable']} ({heavy.iloc[0]['outlier_%']}%). Domain validation needed.")
    else:
        note("Outliers", "Good", "No numeric variable has more than 5% IQR-outliers.")

In [ ]:
if not selected_numeric:
    print("# No numeric columns to plot - skipping boxplots.")
else:
    ncols = min(3, len(selected_numeric))
    nrows = int(np.ceil(len(selected_numeric) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 2.9 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, selected_numeric):
        values = df[c].replace([np.inf, -np.inf], np.nan).dropna()
        ax.boxplot(values, widths=0.6)
        ax.set_title(c, fontsize=10)
        ax.set_xticks([])
    for ax in axes[len(selected_numeric):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 17. Correlation Analysis

**What this section does and why it matters:** pairwise Pearson correlation shows which numeric
variables move together. In an audit it is mainly a *redundancy and consistency* check: a correlation
of exactly 1.0 usually means the same quantity appears twice (e.g. the same series in two units), and an
unexpected near-zero correlation between variables that should be related is a hint that something is
mislabelled.

> **Correlation does not imply causation.** Nothing below should be read as one variable causing another.

In [ ]:
corr_cols = [c for c in plot_numeric][:25]
strong_pairs = pd.DataFrame()

if len(corr_cols) < 2:
    print("# Fewer than two non-constant numeric columns - skipping correlation analysis.")
else:
    corr = df[corr_cols].corr(method="pearson", numeric_only=True)
    plt.figure(figsize=(min(14, 1 + 0.6 * len(corr_cols)), min(12, 1 + 0.5 * len(corr_cols))))
    sns.heatmap(corr, annot=len(corr_cols) <= 12, fmt=".2f", cmap="coolwarm",
                vmin=-1, vmax=1, square=False, cbar_kws={"shrink": 0.7})
    plt.title("Pearson correlation")
    plt.tight_layout()
    plt.show()

    pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().reset_index()
    pairs.columns = ["variable_a", "variable_b", "r"]
    strong_pairs = (pairs[pairs["r"].abs() > STRONG_CORRELATION]
                    .sort_values("r", key=np.abs, ascending=False).reset_index(drop=True))
    if strong_pairs.empty:
        print(f"No variable pair exceeds |r| > {STRONG_CORRELATION}.")
    else:
        display(strong_pairs)
        for _, r in strong_pairs.iterrows():
            direction = "positive" if r["r"] > 0 else "negative"
            print(f"Strong {direction} correlation: {r['variable_a']} <-> {r['variable_b']}: r = {r['r']:.2f}")
        near_perfect = strong_pairs[strong_pairs["r"].abs() > 0.99]
        if len(near_perfect):
            print("\n|r| > 0.99 - check whether these columns are duplicates of the same quantity:")
            for _, r in near_perfect.iterrows():
                print(f"  {r['variable_a']} <-> {r['variable_b']} (r = {r['r']:.4f})")
        print("\nReminder: correlation does NOT imply causation.")

## 18. Plausibility Checks

**What this section does and why it matters:** applies generic, low-assumption sanity rules and splits
the results into two clearly separated groups:

* **Mathematically invalid** — infinite values, latitude outside [-90, 90], longitude outside [-180, 180],
  a share column outside [0, 100], a probability outside [0, 1], a calendar month outside [1, 12].
  These are wrong regardless of domain.
* **Potentially suspicious** — negative values in a quantity that is usually non-negative, extremely
  large magnitudes, or a large block of exact zeros that may encode "no data".

The second group needs domain judgement: negative electricity demand can be a genuine measurement of
net load, and a zero can be a real zero.

In [ ]:
invalid_findings, suspicious_findings = [], []

# --- mathematically invalid -------------------------------------------------
for c in numeric_cols:
    s = pd.to_numeric(df[c], errors="coerce")
    n_inf = int(np.isinf(s.to_numpy(dtype="float64", na_value=np.nan)).sum())
    if n_inf:
        invalid_findings.append(f"{c}: {n_inf:,} infinite values")
    name = str(c).lower()
    if any(k in name for k in ["percent", "_pct", "pct_", "share", "%"]):
        bad = int(((s < 0) | (s > 100)).sum())
        if bad:
            invalid_findings.append(f"{c}: {bad:,} values outside [0, 100] in a percentage-named column")
    if "prob" in name:
        bad = int(((s < 0) | (s > 1)).sum())
        if bad:
            invalid_findings.append(f"{c}: {bad:,} values outside [0, 1] in a probability-named column")
    if name == "month":
        bad = int(((s < 1) | (s > 12)).sum())
        if bad:
            invalid_findings.append(f"{c}: {bad:,} values outside [1, 12]")
    if name == "year":
        bad = int(((s < 1500) | (s > 2200)).sum())
        if bad:
            invalid_findings.append(f"{c}: {bad:,} implausible year values (outside 1500-2200)")

for c in lat_cols:
    s = pd.to_numeric(df[c], errors="coerce")
    bad = int(((s < -90) | (s > 90)).sum())
    if bad:
        invalid_findings.append(f"{c}: {bad:,} latitudes outside [-90, 90]")
for c in lon_cols:
    s = pd.to_numeric(df[c], errors="coerce")
    bad = int(((s < -180) | (s > 180)).sum())
    if bad:
        invalid_findings.append(f"{c}: {bad:,} longitudes outside [-180, 180]")

if main_time_col is not None and time_parsed is not None:
    future = int((time_parsed > pd.Timestamp.today()).sum())
    ancient = int((time_parsed < pd.Timestamp("1800-01-01")).sum())
    if future:
        suspicious_findings.append(f"{main_time_col}: {future:,} timestamps in the future")
    if ancient:
        invalid_findings.append(f"{main_time_col}: {ancient:,} timestamps before 1800")

# --- potentially suspicious -------------------------------------------------
NON_NEGATIVE_HINTS = ["population", "gdp", "capacity", "generation", "production", "consumption",
                      "demand", "count", "number", "distance", "area", "volume", "mass"]
for c in numeric_cols:
    s = pd.to_numeric(df[c], errors="coerce")
    name = str(c).lower()
    n_neg = int((s < 0).sum())
    if n_neg and any(h in name for h in NON_NEGATIVE_HINTS):
        suspicious_findings.append(
            f"{c}: {n_neg:,} negative values in a variable whose name suggests a non-negative quantity "
            f"(min = {s.min():,.4g}) - may be legitimate if it is a net or change quantity")
    finite = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(finite) and finite.abs().max() > 1e12:
        suspicious_findings.append(f"{c}: extreme magnitude, max |value| = {finite.abs().max():.3g}")
    zeros = int((s == 0).sum())
    if len(df) and zeros / len(df) > 0.3 and n_unique(df[c]) > 2:
        suspicious_findings.append(
            f"{c}: {zeros:,} zeros ({zeros/len(df):.1%} of rows) - real zeros or encoded missing data?")

print("=== MATHEMATICALLY INVALID (confirmed problems) ===")
if invalid_findings:
    for f in invalid_findings:
        print("  -", f)
else:
    print("  none detected")

print("\n=== POTENTIALLY SUSPICIOUS (needs domain judgement) ===")
if suspicious_findings:
    for f in suspicious_findings:
        print("  -", f)
else:
    print("  none detected")

if invalid_findings:
    note("Invalid values", "Problem", "; ".join(invalid_findings[:4]))
elif suspicious_findings:
    note("Invalid values", "Warning",
         "No mathematically invalid values; " + str(len(suspicious_findings)) +
         " potentially suspicious pattern(s), e.g. " + suspicious_findings[0])
else:
    note("Invalid values", "Good", "No invalid or suspicious values detected by the generic rules.")

## 19. Unit and Scale Inspection

**What this section does and why it matters:** a number without a unit cannot be validated, compared or
combined with another dataset. This section reports units **only where they are actually documented** —
in NetCDF variable attributes or in the column name itself (`load_mw`, `gdp_usd`, `temperature_c`).

> Units are never guessed and never converted. Variables with no documented unit are flagged as a
> metadata gap, which is a real usability limitation even when the numbers themselves are fine.

In [ ]:
UNIT_TOKENS = {
    "mw": "MW", "mwh": "MWh", "gw": "GW", "gwh": "GWh", "kw": "kW", "kwh": "kWh",
    "twh": "TWh", "tw": "TW", "j": "J", "gj": "GJ", "tj": "TJ", "toe": "toe",
    "usd": "USD", "eur": "EUR", "gbp": "GBP", "dollar": "USD", "euro": "EUR",
    "pct": "%", "percent": "%", "share": "share (unit unclear)",
    "c": "degC", "degc": "degC", "celsius": "degC", "k": "K", "kelvin": "K", "f": "degF",
    "mm": "mm", "cm": "cm", "m": "m", "km": "km", "ms": "m/s", "kmh": "km/h",
    "kg": "kg", "t": "t", "tonnes": "tonnes", "kt": "kt", "mt": "Mt", "gt": "Gt",
    "hpa": "hPa", "pa": "Pa", "ppm": "ppm", "capita": "per capita", "persons": "persons",
    "people": "persons", "inhabitants": "persons",
}

unit_rows = []
for c in df.columns:
    unit, source = None, "not documented"
    if is_netcdf and c in ds.variables and "units" in ds[c].attrs:
        unit, source = str(ds[c].attrs["units"]), "NetCDF metadata (units attribute)"
    else:
        name = str(c).lower()
        inside = re.findall(r"[\(\[]([^)\]]+)[\)\]]", name)     # e.g. "demand (MW)"
        token = inside[0].strip() if inside else (re.split(r"[_\-. ]", name)[-1] if re.split(r"[_\-. ]", name) else "")
        if token in UNIT_TOKENS:
            unit, source = UNIT_TOKENS[token], "column name suffix (inferred from the name, not metadata)"
        elif "%" in str(c):
            unit, source = "%", "column name contains '%'"
    unit_rows.append({"column": c, "unit": unit if unit else "-", "unit_source": source,
                      "long_name": (str(ds[c].attrs.get("long_name", "-"))
                                    if is_netcdf and c in ds.variables else "-")})

units_table = pd.DataFrame(unit_rows)
display(units_table)

documented = units_table[units_table["unit"] != "-"]
numeric_no_unit = [c for c in numeric_cols
                   if c in units_table.loc[units_table["unit"] == "-", "column"].tolist()
                   and str(c).lower() not in {"year", "month", "day", "hour", "id"}]

print("Units found for %d of %d columns." % (len(documented), len(units_table)))
if len(documented):
    for _, r in documented.iterrows():
        print(f"  {r['column']} -> {r['unit']}   [{r['unit_source']}]")
print("\nNumeric variables with NO documented unit (metadata gap, values not questioned):")
print(" ", numeric_no_unit or "none")

In [ ]:
# Do variables sharing a documented unit sit on wildly different scales?
scale_flags = []
doc_numeric = [c for c in numeric_cols if c in set(documented["column"])] if len(documented) else []
by_unit = {}
for c in doc_numeric:
    by_unit.setdefault(units_table.loc[units_table["column"] == c, "unit"].iloc[0], []).append(c)

for unit, cols in by_unit.items():
    if len(cols) < 2:
        continue
    medians = {c: float(df[c].abs().replace(0, np.nan).median()) for c in cols}
    medians = {k: v for k, v in medians.items() if pd.notna(v) and v > 0}
    if len(medians) >= 2 and max(medians.values()) / min(medians.values()) > 1000:
        scale_flags.append(f"unit '{unit}': typical magnitudes differ by >1000x -> "
                           + ", ".join(f"{k}~{v:.3g}" for k, v in medians.items()))

# Columns whose names share a stem but whose ranges differ by orders of magnitude.
stems = {}
for c in numeric_cols:
    stem = re.split(r"[_\-. ]", str(c).lower())[0]
    stems.setdefault(stem, []).append(c)
for stem, cols in stems.items():
    if len(cols) < 2:
        continue
    medians = {c: float(df[c].abs().replace(0, np.nan).median()) for c in cols}
    medians = {k: v for k, v in medians.items() if pd.notna(v) and v > 0}
    if len(medians) >= 2 and max(medians.values()) / min(medians.values()) > 1000:
        scale_flags.append(f"columns starting with '{stem}' differ by >1000x in magnitude -> "
                           + ", ".join(f"{k}~{v:.3g}" for k, v in medians.items()))

if scale_flags:
    print("Possible unit/scale inconsistencies (flagged, nothing converted):")
    for f in scale_flags:
        print("  -", f)
else:
    print("No obvious scale inconsistency between related variables.")

if numeric_no_unit and not len(documented):
    note("Units / metadata", "Problem",
         f"No unit metadata anywhere; {len(numeric_no_unit)} numeric variables undocumented.")
elif numeric_no_unit or scale_flags:
    note("Units / metadata", "Warning",
         f"{len(numeric_no_unit)} numeric variable(s) without documented units"
         + (f"; {len(scale_flags)} possible scale inconsistency." if scale_flags else "."))
else:
    note("Units / metadata", "Good", "Every numeric variable has a documented unit.")

## 20. Internal Consistency

**What this section does and why it matters:** checks whether the dataset agrees with itself. Typical
symptoms of a broken pipeline are one entity carrying two different codes, the same key appearing twice,
identifier columns containing nulls, and "start" dates that fall after "end" dates. These problems break
joins and aggregations even when every individual column looks fine.

Everything is documented, nothing is repaired.

In [ ]:
consistency_issues = []

# 1) Entity identifier columns that disagree with each other (name <-> code mappings).
code_like = [c for c in geo_cols + entity_name_cols if c in cat_cols]
code_like = list(dict.fromkeys(code_like))
for a, b in [(a, b) for i, a in enumerate(code_like) for b in code_like[i+1:]]:
    pair = df[[a, b]].dropna().astype(str).drop_duplicates()
    if pair.empty:
        continue
    a_to_b = pair.groupby(a)[b].nunique()
    b_to_a = pair.groupby(b)[a].nunique()
    bad_a = a_to_b[a_to_b > 1]
    bad_b = b_to_a[b_to_a > 1]
    if len(bad_a):
        consistency_issues.append(
            f"'{a}' maps to several '{b}' values for {len(bad_a)} entities, e.g. "
            f"{bad_a.index[0]} -> {sorted(pair.loc[pair[a] == bad_a.index[0], b].unique())[:4]}")
    if len(bad_b):
        consistency_issues.append(
            f"'{b}' maps to several '{a}' values for {len(bad_b)} entities, e.g. "
            f"{bad_b.index[0]} -> {sorted(pair.loc[pair[b] == bad_b.index[0], a].unique())[:4]}")

# 2) Duplicate entity-time combinations on the main axes.
dup_entity_time = 0
if main_entity_col is not None and main_time_col is not None:
    combo = pd.DataFrame({"entity": df[main_entity_col].astype(str), "time": time_parsed})
    dup_entity_time = int(combo.dropna().duplicated().sum())
    print(f"Duplicate '{main_entity_col} + {main_time_col}' combinations: {dup_entity_time:,}")
    if dup_entity_time:
        consistency_issues.append(
            f"{dup_entity_time:,} duplicate {main_entity_col}+{main_time_col} combinations "
            f"({dup_entity_time / len(df):.2%} of rows)")
        examples = combo.dropna()[combo.dropna().duplicated(keep=False)].head(6)
        display(examples)
else:
    print("# Entity and/or time axis missing - skipping entity-time uniqueness check.")

# 3) Missing identifiers.
for c in set(entity_name_cols + geo_cols):
    n_missing = int(df[c].isna().sum())
    if n_missing:
        consistency_issues.append(f"identifier column '{c}' has {n_missing:,} missing values")

# 4) Date ordering between start/end style columns.
start_cols = [c for c in df.columns if "start" in str(c).lower() or "from" in str(c).lower()]
end_cols = [c for c in df.columns if "end" in str(c).lower() or "to" == str(c).lower()]
for sc in start_cols:
    for ec in end_cols:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            s = pd.to_datetime(df[sc], errors="coerce")
            e = pd.to_datetime(df[ec], errors="coerce")
        bad = int((s > e).sum())
        if bad:
            consistency_issues.append(f"{bad:,} rows where '{sc}' is later than '{ec}' (impossible ordering)")

# 5) Equivalent-looking columns stored with different dtypes.
stem_types = {}
for c in df.columns:
    stem = re.split(r"[_\-. ]", str(c).lower())[0]
    stem_types.setdefault(stem, set()).add(str(df[c].dtype))
for stem, types in stem_types.items():
    if len(types) > 1 and len([c for c in df.columns if str(c).lower().startswith(stem)]) > 1:
        numeric_family = {t for t in types if t.startswith(("int", "float"))}
        if numeric_family and (types - numeric_family):
            consistency_issues.append(f"columns starting with '{stem}' use mixed dtypes {sorted(types)}")

# 6) Reuse of the scale flags from section 19 and of the NetCDF metadata findings.
consistency_issues.extend(scale_flags)
consistency_issues.extend(netcdf_notes)

print("\n=== INTERNAL CONSISTENCY FINDINGS ===")
if consistency_issues:
    for issue in consistency_issues:
        print("  -", issue)
else:
    print("  No internal inconsistency detected by these checks.")

if dup_entity_time or any("impossible ordering" in i for i in consistency_issues):
    note("Consistency", "Problem", "; ".join(consistency_issues[:3]))
elif consistency_issues:
    note("Consistency", "Warning", "; ".join(consistency_issues[:3]))
else:
    note("Consistency", "Good", "Identifier mappings, key uniqueness and date ordering are consistent.")

## 21. Time-Series Quality

**What this section does and why it matters:** for time-indexed data the *sequence* carries information
that column statistics cannot see: whether rows are ordered, whether intervals are regular, where the
long gaps are, and whether the level of a series shifts abruptly. Abrupt jumps often mark a change of
methodology, a unit change or a reporting break — but they can equally be real events.

Consecutive differences are computed on a **derived, sorted copy**; `df` itself is untouched, and jumps
are flagged, not corrected. For panel data the checks run per entity.

In [ ]:
if main_time_col is None or not numeric_cols:
    print("# No time axis and/or no numeric variables - skipping time-series checks.")
    ts_issues = []
else:
    ts_issues = []
    # DERIVED frame for sequence analysis only.
    # One "series" = one entity, or - for gridded NetCDF - one grid cell (the non-time coordinates).
    if main_entity_col is not None:
        series_keys = [main_entity_col]
    elif is_netcdf:
        series_keys = [c for c in df.columns if c in ds.coords and c != main_time_col]
    else:
        series_keys = []

    ts = pd.DataFrame({"time": time_parsed})
    if series_keys:
        ts["entity"] = df[series_keys].astype(str).agg(" | ".join, axis=1)
        print("Series identified by:", series_keys, f"-> {ts['entity'].nunique():,} series")
    else:
        ts["entity"] = "__single_series__"
        if time_parsed.dropna().duplicated().any():
            print("Note: timestamps repeat but no entity/grid key was detected. The checks below "
                  "treat the file as ONE series, which may overstate duplicates and gaps.")
    measurement_cols = [c for c in numeric_cols
                        if c not in constant_cols and c not in structural_cols]
    value_cols = [c for c in important_cols if c in measurement_cols]
    value_cols += [c for c in measurement_cols if c not in value_cols]
    value_cols = value_cols[:5]        # keep the output readable
    for c in value_cols:
        # Infinite values are excluded here so they cannot dominate the jump statistics;
        # they are reported separately in the plausibility section.
        ts[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    ts = ts.dropna(subset=["time"])

    # Ordering
    ordered = ts.groupby("entity")["time"].apply(lambda s: s.is_monotonic_increasing)
    n_unordered = int((~ordered).sum())
    print(f"Series stored in chronological order: {int(ordered.sum())} of {len(ordered)} "
          f"({'all ordered' if n_unordered == 0 else str(n_unordered) + ' NOT ordered'})")
    if n_unordered:
        ts_issues.append(f"{n_unordered} series are not stored in chronological order "
                         f"(not an error, but sequence-sensitive code must sort first)")

    # Duplicate timestamps within an entity
    dup_within = int(ts.duplicated(subset=["entity", "time"]).sum())
    print(f"Duplicate timestamps within a series: {dup_within:,}")
    if dup_within:
        ts_issues.append(f"{dup_within:,} duplicate entity-timestamp pairs")

    # Interval regularity and long gaps per entity
    sorted_ts = ts.sort_values(["entity", "time"])         # derived ordering, df untouched
    gaps = sorted_ts.groupby("entity")["time"].diff().dropna()
    if len(gaps):
        med_gap = gaps.median()
        irregular_share = float(((gaps - med_gap).abs() > 0.2 * med_gap).mean())
        long_gaps = gaps[gaps > 3 * med_gap]
        print(f"Median interval: {med_gap} | steps more than 20% away from it: {irregular_share:.1%}")
        print(f"Gaps longer than 3x the median interval: {len(long_gaps):,}")
        if len(long_gaps):
            worst = sorted_ts.loc[long_gaps.sort_values(ascending=False).head(5).index]
            display(worst[["entity", "time"]].assign(gap=long_gaps.sort_values(ascending=False).head(5).values))
            ts_issues.append(f"{len(long_gaps):,} gaps longer than 3x the median interval")
        if irregular_share > 0.1:
            ts_issues.append(f"irregular sampling: {irregular_share:.1%} of steps deviate "
                             f"more than 20% from the median interval")

In [ ]:
if main_time_col is None or not numeric_cols:
    print("# No time axis - skipping jump/structural-break detection.")
else:
    print("Extreme changes between consecutive observations "
          "(|change| > 10x the median absolute change of that series).")
    print("Flagged for inspection - these are NOT automatically errors.\n")
    jump_rows = []
    for c in value_cols:
        d = sorted_ts.groupby("entity")[c].diff()
        # The threshold is computed per entity, so large and small series are treated fairly.
        med_abs = d.abs().groupby(sorted_ts["entity"]).transform("median")
        usable = med_abs.notna() & (med_abs > 0)
        if not usable.any():
            print(f"  {c}: median absolute change is 0 or undefined - jump test not meaningful.")
            continue
        extreme = usable & (d.abs() > 10 * med_abs)
        n_extreme = int(extreme.sum())
        print(f"  {c}: typical |change| per entity = {med_abs[usable].median():,.4g} "
              f"| extreme jumps = {n_extreme:,}")
        if n_extreme:
            top = d.abs().where(extreme).sort_values(ascending=False).head(3).index
            for i in top:
                jump_rows.append({"variable": c,
                                  "entity": sorted_ts.loc[i, "entity"],
                                  "time": sorted_ts.loc[i, "time"],
                                  "value": sorted_ts.loc[i, c],
                                  "change_vs_previous": d.loc[i]})
            ts_issues.append(f"{c}: {n_extreme:,} extreme consecutive changes (needs domain validation)")
    if jump_rows:
        display(pd.DataFrame(jump_rows))

    # Simple level-shift check per series: mean of the first half vs the second half of the period.
    print("\nLevel comparison per series, first half vs second half of the time range")
    print("(ratio > 5 or < 0.2 suggests a possible structural break, methodology or unit change):\n")
    shift_rows = []
    mid = sorted_ts["time"].min() + (sorted_ts["time"].max() - sorted_ts["time"].min()) / 2
    for c in value_cols:
        for entity, sub in sorted_ts.groupby("entity"):
            first = sub.loc[sub["time"] <= mid, c].mean()
            second = sub.loc[sub["time"] > mid, c].mean()
            if pd.notna(first) and pd.notna(second) and first != 0 and np.isfinite(first):
                ratio = abs(second / first)
                if ratio > 5 or ratio < 0.2:
                    shift_rows.append({"variable": c, "entity": entity,
                                       "mean_first_half": first, "mean_second_half": second,
                                       "ratio": round(ratio, 2)})
    if shift_rows:
        display(pd.DataFrame(shift_rows))
        for r in shift_rows[:5]:
            ts_issues.append(f"{r['variable']} / {r['entity']}: mean level changes by a factor of "
                             f"{r['ratio']} between the first and second half of the period "
                             f"(possible structural break - needs domain validation)")
    else:
        print("  No series changes its mean level by more than a factor of 5.")

In [ ]:
if main_time_col is None:
    print("# No time axis - no time-series quality summary.")
elif ts_issues:
    print("Time-series quality findings:")
    for i in ts_issues:
        print("  -", i)
else:
    print("No time-series structural problems detected: ordered, regular, no long gaps, no extreme jumps.")

## 22. Coverage Summary

**What this section does and why it matters:** brings the availability picture together in one block.
A large row count can hide poor coverage, so the summary always states entities, time span, expected
versus observed periods and the balance of the panel next to the raw totals.

In [ ]:
print("=" * 62)
print("COVERAGE SUMMARY")
print("=" * 62)
print(f"{'File':<28}: {path.name}")
print(f"{'Total records':<28}: {len(df):,}")
print(f"{'Total columns':<28}: {df.shape[1]}")
print(f"{'Unique entities':<28}: "
      f"{n_entities if n_entities is not None else 'no entity column detected'}"
      f"{'  (' + str(main_entity_col) + ')' if main_entity_col else ''}")

if main_time_col is not None:
    values = pd.DatetimeIndex(time_parsed.dropna().unique()).sort_values()
    print(f"{'Time column':<28}: {main_time_col}")
    print(f"{'Time range':<28}: {values.min().date()}  ->  {values.max().date()}")
    print(f"{'Inferred frequency':<28}: {freq_label}")
    print(f"{'Expected periods':<28}: {expected_periods if expected_periods is not None else 'not determinable'}")
    print(f"{'Observed periods':<28}: {observed_periods if observed_periods is not None else len(values)}")
    print(f"{'Missing periods':<28}: {len(missing_periods) if expected_periods is not None else 'not determinable'}")
else:
    print(f"{'Time coverage':<28}: no time column detected")

if main_entity_col is not None:
    per_entity = df[main_entity_col].value_counts()
    print(f"{'Records per entity':<28}: min {per_entity.min():,} | median {per_entity.median():,.0f} "
          f"| mean {per_entity.mean():,.1f} | max {per_entity.max():,}")
    print(f"{'Panel balance':<28}: {balance}")
    if not entity_time_table.empty and "missing_periods_vs_full_range" in entity_time_table:
        incomplete = int((entity_time_table["missing_periods_vs_full_range"] > 0).sum())
        print(f"{'Entities below full range':<28}: {incomplete} of {len(entity_time_table)}")

if is_netcdf:
    print(f"{'NetCDF dimensions':<28}: " + ", ".join(f"{k}={v}" for k, v in ds.sizes.items()))
print(f"{'Overall missing cells':<28}: {overall_missing_pct:.2f}%")
print(f"{'Fully duplicated rows':<28}: {n_dup_rows:,}")
print("=" * 62)
print("A large row count does not imply good coverage - read the entity and period lines above.")

## 23. Data Quality Scorecard

**What this section does and why it matters:** condenses the audit into one table. Each row is a quality
dimension, a status (`Good` / `Warning` / `Problem` / `Not applicable`) and the **objective evidence**
that produced it — collected automatically by the sections above.

There is deliberately **no single overall score**: the dimensions are not commensurable, and their
weight depends entirely on the intended use of the data.

In [ ]:
DIMENSION_ORDER = ["Dataset size", "Missing data", "Duplicates", "Temporal coverage",
                   "Entity coverage", "Data types", "Constant columns", "Outliers",
                   "Invalid values", "Units / metadata", "Consistency"]

scorecard = pd.DataFrame(findings).drop_duplicates(subset=["quality_dimension"], keep="last")
missing_dims = [d for d in DIMENSION_ORDER if d not in set(scorecard["quality_dimension"])]
if missing_dims:
    scorecard = pd.concat([scorecard, pd.DataFrame(
        [{"quality_dimension": d, "status": "Not applicable",
          "evidence": "Section skipped - the required column type is absent."} for d in missing_dims])])

scorecard["order"] = scorecard["quality_dimension"].apply(
    lambda d: DIMENSION_ORDER.index(d) if d in DIMENSION_ORDER else 99)
scorecard = scorecard.sort_values("order").drop(columns="order").reset_index(drop=True)

with pd.option_context("display.max_colwidth", 130):
    display(scorecard)

counts = scorecard["status"].value_counts()
print("Status counts:", counts.to_dict())
print("\nStatus meaning:")
print("  Good     - the check found no issue in this dimension.")
print("  Warning  - something needs a decision or domain validation before use.")
print("  Problem  - a confirmed defect in the data as delivered.")
print("  n/a      - the check does not apply to this dataset.")

## 24. Final Dataset Audit

**What this section does and why it matters:** the written summary of everything above — size, missing
data, duplicates, coverage, problems, strengths and research relevance. It is generated from the values
computed in this notebook, so it stays consistent with the evidence.

The **Recommendation** field is intentionally left empty: the keep / reject / investigate-further
decision belongs to the analyst, not to the audit.

In [ ]:
lines = []
add = lines.append

add("=" * 70)
add("FINAL DATASET AUDIT — " + path.name)
add("=" * 70)

add("\nDATASET SIZE")
add(f"  Rows        : {len(df):,}")
add(f"  Columns     : {df.shape[1]}")
add(f"  Entities    : {n_entities if n_entities is not None else 'no entity column detected'}"
    f"{' (' + str(main_entity_col) + ')' if main_entity_col else ''}")
if main_time_col is not None:
    tvals = pd.DatetimeIndex(time_parsed.dropna().unique()).sort_values()
    add(f"  Time range  : {tvals.min().date()} -> {tvals.max().date()} ({freq_label})")
else:
    add("  Time range  : no time column detected")
add(f"  File size   : {file_size_mb:.2f} MB")
if partial_read:
    add(f"  PARTIAL AUDIT: {partial_reason}")
    add("                 every number in this report describes that sample only.")
elif partial_reason:
    add(f"  Note        : {partial_reason}")

add("\nMISSING DATA")
add(f"  Overall missing percentage : {overall_missing_pct:.2f}%")
significant = missing_table[missing_table['missing_%'] > 10]
add(f"  Columns with >10% missing  : {len(significant)}"
    + (": " + ", ".join(f"{r['column']} ({r['missing_%']:.1f}%)" for _, r in significant.head(8).iterrows())
       if len(significant) else ""))
add(f"  Maximum column missingness : {max_missing:.2f}%"
    + (f" ({missing_table.iloc[0]['column']})" if len(missing_table) else ""))
add(f"  Rows with >=1 missing value: {rows_with_missing:,} ({rows_with_missing / len(df) * 100:.2f}%)"
    f" | ignoring fully empty columns: {rows_missing_excl_empty:,} "
    f"({rows_missing_excl_empty / len(df) * 100:.2f}%)"
    if len(df) else "  Rows with >=1 missing value: n/a")

add("\nDUPLICATES")
add(f"  Fully duplicated rows            : {n_dup_rows:,} ({dup_pct:.2f}%)")
add(f"  Duplicate entity-time combinations: "
    f"{dup_entity_time:,}" if main_entity_col and main_time_col else
    "  Duplicate entity-time combinations: not testable (entity and/or time column missing)")

add("\nCOVERAGE")
add(f"  Number of entities  : {n_entities if n_entities is not None else 'n/a'}")
add(f"  Temporal frequency  : {freq_label if main_time_col else 'n/a'}")
if expected_periods is not None:
    add(f"  Expected periods    : {expected_periods:,} | observed: {observed_periods:,} | "
        f"missing: {len(missing_periods):,}")
add(f"  Coverage consistency: {balance}")

# ---- problems and strengths, derived from the collected evidence -------------
problems, strengths = [], []

if empty_cols:
    problems.append(f"{len(empty_cols)} column(s) are 100% empty: {', '.join(map(str, empty_cols[:5]))}")
for _, r in missing_table[missing_table["missing_%"] > 25].iterrows():
    if r["column"] not in empty_cols:
        problems.append(f"High missingness in '{r['column']}' ({r['missing_%']:.1f}%)")
if n_dup_rows:
    problems.append(f"{n_dup_rows:,} fully duplicated rows ({dup_pct:.2f}%)")
if main_entity_col and main_time_col and dup_entity_time:
    problems.append(f"{dup_entity_time:,} duplicate {main_entity_col}+{main_time_col} combinations")
if expected_periods is not None and missing_periods:
    problems.append(f"{len(missing_periods)} of {expected_periods} expected periods are absent "
                    f"(e.g. {', '.join(missing_periods[:5])})")
if not entity_time_table.empty and "missing_periods_vs_full_range" in entity_time_table:
    incomplete = int((entity_time_table["missing_periods_vs_full_range"] > 0).sum())
    if incomplete:
        problems.append(f"{incomplete} of {len(entity_time_table)} entities do not cover the full time range")
if 'type_issue_table' in dir() and not type_issue_table.empty:
    for _, r in type_issue_table[~type_issue_table["issue"].str.contains("informational")].head(6).iterrows():
        problems.append(f"'{r['column']}': {r['issue']} ({r['evidence']})")
for issue in invalid_findings:
    problems.append(f"Invalid values - {issue}")
for issue in geo_issues[:6]:
    problems.append(f"Identifier inconsistency - {issue}")
for issue in consistency_issues[:6]:
    problems.append(f"Consistency - {issue}")
if constant_cols:
    problems.append(f"Constant column(s) carrying no information: {', '.join(map(str, constant_cols[:6]))}")
if numeric_no_unit:
    problems.append(f"{len(numeric_no_unit)} numeric variable(s) have no documented unit: "
                    f"{', '.join(map(str, numeric_no_unit[:6]))}")
for issue in suspicious_findings[:6]:
    problems.append(f"Potentially suspicious - {issue}")
if len(outlier_table):
    heavy = outlier_table[outlier_table["outlier_%"] > 5]
    if len(heavy):
        problems.append("Potential extreme outliers (domain validation needed) in: "
                        + ", ".join(f"{r['variable']} ({r['outlier_%']}%)" for _, r in heavy.head(5).iterrows()))
if main_time_col is not None and ts_issues:
    for issue in ts_issues[:6]:
        problems.append(f"Time series - {issue}")

if len(df) >= 10000:
    strengths.append(f"Large dataset ({len(df):,} rows)")
if overall_missing_pct < 1:
    strengths.append(f"Very low missingness ({overall_missing_pct:.2f}% of cells)")
elif overall_missing_pct < 5:
    strengths.append(f"Low missingness ({overall_missing_pct:.2f}% of cells)")
if n_dup_rows == 0:
    strengths.append("No fully duplicated rows")
if main_entity_col and main_time_col and dup_entity_time == 0:
    strengths.append(f"Unique {main_entity_col}+{main_time_col} key (clean panel grain)")
if expected_periods is not None and not missing_periods:
    strengths.append(f"Complete {freq_label} calendar with no missing periods")
if n_entities and n_entities >= 10:
    strengths.append(f"Broad entity coverage ({n_entities} entities)")
if balance.startswith("balanced"):
    strengths.append("Balanced panel: every entity has the same number of records")
if main_time_col is not None:
    tvals = pd.DatetimeIndex(time_parsed.dropna().unique())
    span_years = (tvals.max() - tvals.min()).days / 365.25
    if span_years >= 5:
        strengths.append(f"Long historical period ({span_years:.1f} years)")
if len(documented):
    strengths.append(f"Units documented for {len(documented)} column(s)")
if is_netcdf and ds.attrs:
    strengths.append("NetCDF global metadata present (source/description attributes)")
if not invalid_findings:
    strengths.append("No mathematically invalid values detected")
if not constant_cols and not near_constant_cols:
    strengths.append("Every column carries variation")
if geo_cols and not geo_issues:
    strengths.append("Geographic identifiers are internally consistent")

add("\nPOTENTIAL PROBLEMS")
if problems:
    for p in problems:
        add(f"  - {p}")
else:
    add("  - None detected by the checks in this notebook.")

add("\nSTRENGTHS")
if strengths:
    for s in strengths:
        add(f"  + {s}")
else:
    add("  + None of the positive criteria in this notebook were met.")

add("\nRESEARCH / MODELLING RELEVANCE (objective observations only)")
add(f"  - Structure: {'panel (entity x time)' if main_entity_col and main_time_col else ('time series' if main_time_col else ('cross-section' if main_entity_col else 'flat table'))}"
    f" with {len(numeric_cols)} numeric, {len(cat_cols)} categorical and {len(datetime_cols)} datetime columns.")
if important_cols:
    add(f"  - Columns whose names suggest substantive content: {', '.join(map(str, important_cols[:12]))}"
        + (" ..." if len(important_cols) > 12 else ""))
if main_time_col and n_entities:
    add(f"  - Supports entity-level and time-based analysis for {n_entities} entities across "
        f"{observed_periods if observed_periods else 'the observed'} periods, subject to the coverage gaps listed above.")
add("  - Before any research or ML use, verify: variable definitions and units against the source "
    "documentation, the meaning of zeros/negatives, the reason for missing periods, and the "
    "provenance of any duplicated keys.")

add("\nRECOMMENDATION")
add("  (left blank by design - the keep / reject / investigate decision is yours)")
add("=" * 70)

print("\n".join(lines))

### Recommendation

*Left intentionally blank — record your own keep / reject / investigate-further decision here.*

---

**Scope reminder:** this notebook only *audits*. It never dropped a row or column, filled a value,
interpolated a gap, renamed a field, converted a unit, removed an outlier, deduplicated or aggregated.
Any transformation shown above (parsed dates, sorted copies, string samples) lived in a clearly named
derived object; the loaded `df` (and `ds` for NetCDF) is exactly what the file contained.

## 25. Save the Report / Batch Mode (one report per dataset folder)

**What this section does and why it matters:** an audit is only useful if you can read it later and
compare it with the other files you downloaded. Two things happen here:

1. **This file** — its final audit text and scorecard are saved to `REPORT_ROOT` as a `.txt`
   (plus a `.json` summary when the batch runner asks for one).
2. **Everything else** — `batch_audit.py` runs this whole notebook once per file in
   `DATA_SOURCES` and saves the results grouped the way your folders are grouped:

```text
reports/
  overview.md                              all sources side by side
  00_file_inventory.csv                    every file found, audited or skipped, and why
  00_duplicate_files.csv                   byte-identical downloads
  ENTSOE/
    index.md                               all ENTSO-E datasets side by side
    MonthlyDomesticValues/
      report.html                          every output of every cell, for all 8 files
      summary.md                           file table, schema comparison, audit per file
      audits/monthly_domestic_values_2019.json
    InventoryofGeneration/ ...
  OWID/
    report.html   summary.md               (files that sit directly in the source folder)
```

A dataset folder that holds several yearly files (ENTSO-E, IRENASTAT) gets **one** report covering all
of them, with a schema comparison across the years — that is where inconsistent columns between
downloads show up.

Runtime is roughly a few seconds per file plus load time, so a full run over all sources takes a while;
set `RUN_BATCH = False` in the configuration cell to skip it, or run
`python batch_audit.py --sources ENTSOE` in a terminal for a single source.

In [ ]:
import json

report_dir = Path(REPORT_ROOT)
report_dir.mkdir(parents=True, exist_ok=True)

# Everything the report needs, collected from the variables computed above.
time_min = time_max = None
if main_time_col is not None:
    _tvals = pd.DatetimeIndex(time_parsed.dropna().unique()).sort_values()
    time_min, time_max = str(_tvals.min()), str(_tvals.max())

audit_summary = {
    "file": path.name,
    "path": str(path.resolve()),
    "file_type": ext,
    "file_size_mb": round(file_size_mb, 3),
    "loader": loader,
    "partial_read": bool(partial_read),
    "partial_reason": partial_reason,
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "column_names": [str(c) for c in df.columns],
    "dtypes": {str(c): str(t) for c, t in df.dtypes.items()},
    "n_numeric": len(numeric_cols),
    "n_categorical": len(cat_cols),
    "n_datetime": len(datetime_cols),
    "missing_overall_pct": round(float(overall_missing_pct), 3),
    "missing_worst_column": (str(missing_table.iloc[0]["column"]) if len(missing_table) else None),
    "missing_worst_pct": round(float(max_missing), 3),
    "columns_with_missing": int(cols_with_missing),
    "rows_with_missing": int(rows_with_missing),
    "duplicate_rows": int(n_dup_rows),
    "duplicate_rows_pct": round(float(dup_pct), 3),
    "duplicate_entity_time": int(dup_entity_time),
    "entity_column": (str(main_entity_col) if main_entity_col else None),
    "n_entities": (int(n_entities) if n_entities is not None else None),
    "panel_balance": balance,
    "time_column": (str(main_time_col) if main_time_col else None),
    "time_min": time_min,
    "time_max": time_max,
    "frequency": (freq_label if main_time_col else None),
    "expected_periods": expected_periods,
    "observed_periods": observed_periods,
    "missing_periods": len(missing_periods) if expected_periods is not None else None,
    "constant_columns": [str(c) for c in constant_cols],
    "numeric_columns_without_units": [str(c) for c in numeric_no_unit],
    "problems": problems,
    "strengths": strengths,
    "scorecard": scorecard.to_dict(orient="records"),
    "final_audit_text": "\n".join(lines),
}

if SAVE_TEXT_REPORT:
    text_path = report_dir / f"{path.stem}_audit.txt"
    with open(text_path, "w") as fh:
        fh.write("\n".join(lines))
        fh.write("\n\nDATA QUALITY SCORECARD\n")
        fh.write(scorecard.to_string(index=False))
        fh.write("\n")
    print("Saved text report :", text_path)

if SUMMARY_JSON_PATH:
    Path(SUMMARY_JSON_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(SUMMARY_JSON_PATH, "w") as fh:
        json.dump(audit_summary, fh, indent=2, default=str)
    print("Saved JSON summary:", SUMMARY_JSON_PATH)

In [ ]:
# Batch: audit every file of every source and save one report per dataset folder.
import subprocess
import sys

available = [source for source in DATA_SOURCES if (Path(PROJECT_ROOT) / source).exists()]

if BATCH_CHILD:
    print("# This run was started by the batch runner - not starting a nested batch.")
elif not RUN_BATCH:
    print("# RUN_BATCH is False - skipping the batch. Run it later with:")
    print(f"#   python batch_audit.py --root {PROJECT_ROOT} --out {REPORT_ROOT} "
          f"--sources {' '.join(DATA_SOURCES)}")
elif not available:
    print("# None of the folders in DATA_SOURCES exist here - skipping the batch.")
    print("# Expected:", ", ".join(str(Path(PROJECT_ROOT) / s) for s in DATA_SOURCES))
else:
    to_audit = int(inventory["status"].eq("audit").sum()) if len(inventory) else 0
    print(f"Auditing {to_audit} file(s) across {len(available)} source(s): {', '.join(available)}")
    print("This runs the whole notebook once per file - expect a few seconds each.\n")
    command = [sys.executable, "batch_audit.py", "--root", str(PROJECT_ROOT),
               "--out", REPORT_ROOT, "--notebook", "dataset_inspection.ipynb",
               "--sources", *available]
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout[-12000:])
    if result.stderr.strip():
        print("stderr:\n", result.stderr[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"batch_audit.py failed with exit code {result.returncode}")